In [1]:
# Developed with assistance from Claude (Anthropic).
# Depo ayarı: .gitignore — ham/temiz veri, ara dosyalar, Jupyter geçici dosyaları ve gizli dosyalar depoya gitmez.
from pathlib import Path

KOK = Path.home() / "orbitrary"
gi = KOK / ".gitignore"
kurallar = [
    "# Orbitrary: depoya girmeyecekler (belge 17.11: depoda ham veri, _netrc, şifre ya da token yok)",
    "veri/", "temiz/", "ara/",
    ".virtual_documents/", ".ipynb_checkpoints/", "**/.ipynb_checkpoints/",
    "*.part", "_netrc", ".netrc", "*.token", ".env",
]
eski = gi.read_text(encoding="utf-8").splitlines() if gi.exists() else []
yeni = eski + [k for k in kurallar if k not in eski]
gi.write_text("\n".join(yeni) + "\n", encoding="utf-8")
print(gi.read_text(encoding="utf-8"))

# depoya gidecek dosyaların kaba sayımı (yok sayılanlar hariç)
haric = ("veri", "temiz", "ara", ".git", ".virtual_documents")
kalan = [p for p in KOK.rglob("*") if p.is_file() and p.relative_to(KOK).parts[0] not in haric
         and ".ipynb_checkpoints" not in p.parts]
buyuk = [(str(p.relative_to(KOK)), round(p.stat().st_size / 1e6, 1)) for p in kalan if p.stat().st_size > 50e6]
print(f"Depoya gidecek dosya sayısı (yaklaşık): {len(kalan)}")
print("50 MB'tan büyük olanlar:", buyuk if buyuk else "yok")

# Orbitrary: depoya girmeyecekler (belge 17.11: depoda ham veri, _netrc, şifre ya da token yok)
veri/
temiz/
ara/
.virtual_documents/
.ipynb_checkpoints/
**/.ipynb_checkpoints/
*.part
_netrc
.netrc
*.token
.env

Depoya gidecek dosya sayısı (yaklaşık): 40
50 MB'tan büyük olanlar: yok


In [2]:
# Developed with assistance from Claude (Anthropic).
# P0 (belge 17.5) — sağlama kısmı: meta.json SHA-256'ları ↔ kayit\D1_dosyalar.csv ve temiz\ dosyaları.
# D4 kısmı (Bölüm 8.2 pytest) modüller yazılınca kos.py P0 adımında kapanır.
import json, re, hashlib, time, datetime, platform
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import pandas as pd

KOK = Path.home() / "orbitrary"
TEMIZ, KAYIT = KOK / "temiz", KOK / "kayit"
HEX64 = re.compile(r"^[0-9a-fA-F]{64}$")
BEKLENEN = {"dunya": [f"E{i}" for i in range(1, 11)],
            "ay":    [f"M{i}" for i in range(1, 8)],
            "mars":  [f"R{i}" for i in range(1, 8)]}
t0 = time.time()

# 1) 24 set klasörü ve meta.json
setler = {}
for govde, kodlar in BEKLENEN.items():
    for kod in kodlar:
        aday = sorted(p for p in (TEMIZ / govde).glob(f"{kod}_*") if p.is_dir())
        assert len(aday) == 1, f"{govde}/{kod}: {len(aday)} klasör bulundu {aday}"
        assert (aday[0] / "meta.json").is_file(), f"{aday[0]} meta.json yok"
        setler[f"{govde}/{kod}"] = aday[0]
assert len(setler) == 24
print("24/24 set klasörü ve meta.json var")

# 2) meta.json içindeki bütün SHA-256 değerleri (anahtar yolu ile)
def hexler(o, yol=""):
    if isinstance(o, dict):
        for k, v in o.items(): yield from hexler(v, f"{yol}/{k}")
    elif isinstance(o, list):
        for i, v in enumerate(o): yield from hexler(v, f"{yol}[{i}]")
    elif isinstance(o, str) and HEX64.match(o.strip()):
        yield yol, o.strip().lower()

meta_hex = {}
for ad, kl in setler.items():
    with open(kl / "meta.json", encoding="utf-8") as f:
        meta_hex[ad] = list(hexler(json.load(f)))

# 3) D1_dosyalar.csv içindeki bütün SHA-256 değerleri (sütun adı tahmin edilmez)
d1 = pd.read_csv(KAYIT / "D1_dosyalar.csv", dtype=str, encoding="utf-8-sig")
d1_hex = {v.strip().lower() for v in d1.to_numpy().ravel() if isinstance(v, str) and HEX64.match(v.strip())}
print(f"D1_dosyalar.csv: {len(d1)} satır, {len(d1_hex)} farklı SHA-256; sütunlar: {list(d1.columns)}")

# 4) temiz\ dosyalarının SHA-256'sı (meta.json hariç), paralel
def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        while b := f.read(8 << 20): h.update(b)
    return h.hexdigest()

dosyalar = sorted(p for kl in setler.values() for p in kl.rglob("*") if p.is_file() and p.name != "meta.json")
toplam_gb = sum(p.stat().st_size for p in dosyalar) / 1e9
print(f"temiz\\: {len(dosyalar)} dosya, {toplam_gb:.2f} GB hesaplanıyor...")
with ThreadPoolExecutor(max_workers=4) as ex:
    temiz_sha = dict(zip(dosyalar, ex.map(sha, dosyalar)))
pd.DataFrame({"dosya": [str(p.relative_to(KOK)) for p in temiz_sha],
              "bayt": [p.stat().st_size for p in temiz_sha],
              "sha256": list(temiz_sha.values())}).to_csv(KAYIT / "P0_temiz_sha.csv", index=False, encoding="utf-8")
temiz_hex = set(temiz_sha.values())
print(f"temiz\\ SHA-256 bitti: {time.time()-t0:.0f} s")

# 5) Eşleştirme: her meta.json SHA'sı ya D1'de (ham) ya temiz\'de olmalı; her sette en az 1 ham eşleşme
sonuc, hepsi_ok = {}, True
for ad, liste in meta_hex.items():
    ham = [y for y, h in liste if h in d1_hex]
    tmz = [y for y, h in liste if h in temiz_hex and h not in d1_hex]
    yok = [{"anahtar": y, "sha256": h} for y, h in liste if h not in d1_hex and h not in temiz_hex]
    ok = len(liste) > 0 and len(ham) > 0 and not yok
    hepsi_ok &= ok
    sonuc[ad] = {"klasor": str(setler[ad].relative_to(KOK)), "meta_sha_sayisi": len(liste),
                 "ham_eslesen": len(ham), "temiz_eslesen": len(tmz), "eslesmeyen": yok,
                 "durum": "PASS" if ok else "FAIL"}
    print(f"{ad:10s} {sonuc[ad]['durum']}  meta {len(liste):3d} | ham {len(ham):3d} | temiz {len(tmz):3d} | eşleşmeyen {len(yok)}")

kayit = {"adim": "P0-sağlama", "zaman": datetime.datetime.now().isoformat(timespec="seconds"),
         "makine": platform.node(), "sure_s": round(time.time() - t0, 1),
         "d1_satir": len(d1), "d1_sha_sayisi": len(d1_hex),
         "temiz_dosya": len(dosyalar), "temiz_gb": round(toplam_gb, 3),
         "setler": sonuc, "sağlama_durum": "PASS" if hepsi_ok else "FAIL",
         "D4": "bekliyor (Bölüm 8.2 testleri modüllerle yazılınca kos.py P0'da kapanır)"}
(KAYIT / "P0.json").write_text(json.dumps(kayit, ensure_ascii=False, indent=2), encoding="utf-8")
with open(KAYIT / "14_kasim_log.txt", "a", encoding="utf-8") as f:
    f.write(f"{kayit['zaman']}\tP0-sağlama\t{kayit['sağlama_durum']}\t{kayit['sure_s']} s\t"
            f"{sum(s['durum']=='PASS' for s in sonuc.values())}/24\n")
print(f"\nP0 sağlama: {kayit['sağlama_durum']}  ({sum(s['durum']=='PASS' for s in sonuc.values())}/24)  "
      f"→ kayit\\P0.json, kayit\\P0_temiz_sha.csv  ({kayit['sure_s']} s)")

24/24 set klasörü ve meta.json var
D1_dosyalar.csv: 414 satır, 414 farklı SHA-256; sütunlar: ['hedef', 'set', 'bayt', 'rapor_bayt', 'sha256', 'durum', 'not']
temiz\: 742 dosya, 33.21 GB hesaplanıyor...
temiz\ SHA-256 bitti: 12 s
dunya/E1   PASS  meta 288 | ham 288 | temiz   0 | eşleşmeyen 0
dunya/E2   PASS  meta   1 | ham   1 | temiz   0 | eşleşmeyen 0
dunya/E3   PASS  meta   1 | ham   1 | temiz   0 | eşleşmeyen 0
dunya/E4   PASS  meta  12 | ham  12 | temiz   0 | eşleşmeyen 0
dunya/E5   PASS  meta  12 | ham  12 | temiz   0 | eşleşmeyen 0
dunya/E6   PASS  meta  12 | ham  12 | temiz   0 | eşleşmeyen 0
dunya/E7   PASS  meta   1 | ham   1 | temiz   0 | eşleşmeyen 0
dunya/E8   PASS  meta   1 | ham   1 | temiz   0 | eşleşmeyen 0
dunya/E9   PASS  meta   2 | ham   2 | temiz   0 | eşleşmeyen 0
dunya/E10  PASS  meta   4 | ham   4 | temiz   0 | eşleşmeyen 0
ay/M1      PASS  meta   3 | ham   3 | temiz   0 | eşleşmeyen 0
ay/M2      PASS  meta   1 | ham   1 | temiz   0 | eşleşmeyen 0
ay/M3      PASS

In [3]:
# Developed with assistance from Claude (Anthropic).
# 17.4 sıra 1: kod\ayar.json + kod\ayar.py + testler (Bölüm 8.4, 4.1, 7.3; v1.2.1 madde 4)
import json, sys, subprocess, datetime, textwrap
from pathlib import Path

KOK = Path.home() / "orbitrary"
KOD, KAYIT = KOK / "kod", KOK / "kayit"
(KOD / "testler").mkdir(exist_ok=True)

def P(deger, kaynak, birim=None, bolum=None, durum=None):
    d = {"deger": deger, "kaynak": kaynak}
    if birim: d["birim"] = birim
    if bolum: d["bolum"] = bolum
    if durum: d["durum"] = durum
    return d

ayar_json = {
  "surum": "1.2.1",
  "aciklama": "Bölüm 8.4'teki bütün tasarım parametreleri + Bölüm 4.1 sabitleri. Her değerin yanında kaynağı ya da 'proje kararı'.",
  "parametreler": {
    "tohum": P(20261113, "belge 8.4 / V2", bolum="8.4"),
    "alt_tohum": P({"R_orneklem": 1, "V2": 2, "V3": 3, "V4_ikinci_R": 4, "V5": 5, "V7": 7, "P_fB": 10, "katman_B_R": 11},
                   "proje kararı (belge 17: aşama başına sabit alt tohum)", bolum="17"),
    "cisim": {
      "yaricap_km": P({"dunya": 6371.0, "ay": 1737.4, "mars": 3389.5}, "[73, 3]", "km", "4.1"),
      "eksen_egikligi_der": P({"dunya": 23.44, "ay": 1.54, "mars": 25.19}, "[3, 65]", "derece", "4.1"),
      "sigma_v_m": P({"dunya": 7.17, "ay": 10.0, "mars": 1.0},
                     "Dünya [59] ETOPO sayfası; Ay [105] Smith ve ark. 2010 özeti '~10 m radial accuracy'; "
                     "Mars [104] Smith ve ark. 2001 özeti 'vertical and radial accuracies of ~1 m' (Ay/Mars değeri v1.2.1'de eklendi)",
                     "m", "4.1"),
    },
    "izgara": {
      "Delta_km": P(20.0, "Artemis bölge ölçeği [84]", "km", "4.2"),
      "L_km": P(20.0, "Artemis bölge ölçeği [84]", "km", "4.1"),
      "delta_m": P(500.0, "MOLA 463 m'nin üstü", "m", "4.1"),
      "N": P(40, "N = L/δ", bolum="4.1"),
      "n_komsu": P(20, "proje kararı; kontrol V7", bolum="6.1"),
      "kenar_genisletme_hucre": P(2, "belge 6.1 (kenar hücreleri için 2 hücre dışarı)", bolum="6.1"),
    },
    "uzaklik": {
      "M_yuzdelik": P(1000, "yakınsama (8.4)", bolum="5.1"),
      "M_theta": P(50, "yakınsama (8.4)", bolum="6.2"),
      "P_fB_en_fazla_piksel": P(1000000, "belge 5.2 (en fazla 10⁶ rastgele piksel)", bolum="5.2"),
    },
    "referans": {
      "katman_A": P({"1": 2000, "2": 200, "3": "E'", "4": "E'", "5": "E'", "6": "E'", "7": "E'", "8": 200}, "belge 5.4; kontrol V4", bolum="5.4"),
      "katman_B": P({"1": 2000, "2": 200, "3": 2000, "4": 2000, "5": 2000, "6": 2000, "7": 2000, "8": 200}, "belge 3.2; kontrol V4", bolum="3.2"),
      "en_kucuk": P(100, "belge 5.4 (|R_k| < 100 ise s = null)", bolum="5.4"),
    },
    "topografya": {
      "olcekler": P([1, 4], "belge 6.1 (500 m ve 2 km)", bolum="6.1"),
      "IQR_yuzdelikleri": P([0.25, 0.75], "belge 6.1 [96]", bolum="6.1"),
    },
    "topoloji": {
      "kappa": P(2, "proje kararı; kontrol V7", bolum="6.2"),
      "aday_sayisi": P(30, "proje kararı; kontrol V5", bolum="7.4"),
    },
    "onele": {
      "K_on_adaylari": P([100, 200, 400, 800], "belge 7.1, v1.2 D8", bolum="7.1"),
      "K_on": P(None, "V5: recall ≥ 0,9 sağlayan en küçük; hepsinde < 0,9 ise 800 (K3)", bolum="7.1", durum="V5 ile belirlenir"),
      "recall_esigi": P(0.9, "belge 8.3 V5", bolum="8.3"),
    },
    "maske": {"kara_orani": P(0.9, "proje kararı; kontrol V1", bolum="7.1")},
    "kapi": {
      "mobility": {"egim_der": P(20.0, "[17] NASA Artemis III EVA eğim gereksinimi", "derece", "7.2"),
                   "hucre_orani": P(0.95, "proje kararı; kontrol V1", bolum="7.2"),
                   "olcek_m": P(500.0, "belge 7.2 (500 m Horn eğimi)", "m", "7.2")},
      "buz": {"ana": P(0.90, "[12] IPA sürekli %90–100", bolum="7.2"),
              "yedek": P(0.50, "[12] IPA süreksiz %50–90", bolum="7.2")},
      "astro": {"ana": P(0.03, "[131] aşırı kurak < 0,03 (temiz\\ AI zaten ham/10 000)", bolum="7.2"),
                "yedek": P(0.20, "[131] kurak < 0,20", bolum="7.2")},
      "jeoloji": {"Pv_esigi": P(0.5, "[137] Sobrino ve ark.", bolum="7.2"),
                  "NDVI_ciplak": P(0.2, "[137]", bolum="7.2"),
                  "NDVI_tam": P(0.5, "[137]", bolum="7.2"),
                  "kar_buz_siniri": P(0.5, "proje kararı", bolum="7.2"),
                  "X": P(None, "leave-one-out, V1 Geology çiftleri (v1.2 D7)", bolum="7.2", durum="LOO ile hesaplanır")},
      "yedek_siniri_pencere": P(100, "belge 5.4 ile aynı", bolum="7.2"),
    },
    "uygunluk": {"zorunlu_kategoriler": P([1, 2], "proje kararı; kontrol V8", bolum="7.2"),
                 "en_fazla_eksik": P(1, "proje kararı; kontrol V8", bolum="7.2")},
    "agirlik": {
      "ana_pay": P(0.80, "[84] uyarlaması (80/20); duyarlılık, V1", bolum="7.3"),
      "ana_kategoriler": P({"mobility": [1, 2, 7], "geology": [2, 3, 4], "ice": [5, 6, 8], "astro": [4, 5, 6]},
                           "belge 7.3 tablosu (kalın değerler)", bolum="7.3"),
      "duyarlilik_sema": P(["esit", "geometrik"], "belge 7.4", bolum="7.4"),
    },
    "jeoloji": {"glim_alt_ornekleme": P(5, "proje kararı (hücre başına 5 × 5 nokta)", bolum="4.4"),
                "mars_yayla": P({"volkanik": 0.5, "tortul": 0.5}, "proje kararı; kontrol V8", bolum="6.3")},
    "mineraloji": {"gecerlilik_orani": P(0.5, "proje kararı", bolum="6.4")},
    "termal": {"ay_tgece_gcp_sinir_der": P(70.0, "v1.2 D5", "derece", "6.5"),
               "ay_prp_sinir_der": P(76.0, "belge 6.5 tablosu", "derece", "6.5"),
               "ay_gcp_yerel_saat": P([18.0, 6.0], "v1.2 D5 (18:00–06:00 ortalaması)", "saat", "6.5"),
               "ay_tbol_ust_K": P(250.0, "belge 6.5 (≥ 250 K veri yok)", "K", "6.5")},
    "su_buz": {"islak_esigi": P(0.5, "proje kararı", bolum="6.6"),
               "kar_buz_siniri": P(0.5, "belge 6.6 sıfır kuralı", bolum="6.6"),
               "ay_toryum_ppm": P(4.56, "proje kararı (Ay'ın en yüksek %10'u)", "ppm", "6.6"),
               "ay_prp_kapsama_der": P(75.9, "belge 6.6", "derece", "6.6")},
    "aydinlanma": {"ay_pgda_sinir_der": P(65.0, "PGDA haritasının tam kapsadığı çember", "derece", "6.8"),
                   "Ls_adim": P(3600, "proje kararı (yıllık ortalamanın sayısal integrali)", bolum="6.8")},
    "siralama": {"top3_en_az_km": P(100.0, "proje kararı", "km", "7.4")},
    "katman_B": {"sure_siniri_sa": P(10.0, "proje kararı (K1)", "saat", "11"),
                 "katman_A_sure_siniri_sa": P(4.0, "K2", "saat", "11")},
    "dogrulama": {
      "alfa": P(0.05, "belge 8.3", bolum="8.3"),
      "V2_sinif_orani": P(0.90, "v1.2 D6 (sınıf 2 ve 12 ≥ %90)", bolum="8.3"),
      "V3_bootstrap": P(2000, "belge 8.3 [28]", bolum="8.3"),
      "V3_rastgele_pencere": P(1000, "belge 8.3", bolum="8.3"),
      "V4_hedef": P(200, "belge 8.3", bolum="8.3"),
      "V4_tau_esigi": P(0.9, "belge 8.3, K4", bolum="8.3"),
      "V5_hedef": P(200, "belge 8.3", bolum="8.3"),
      "V6_rs_esigi": P(0.8, "belge 8.3, K5/K6", bolum="8.3"),
      "V7_tekrar": P(20, "belge 8.3", bolum="8.3"),
    },
    "analog_uzaklik_km": P({"bilinen_alti": 20.0, "yakin_ust": 100.0},
                           "proje talimatı v1.2.1 madde 4 (Known < 20, Near 20–100, New > 100 km); puana girmez; D8 ile kilitlenir",
                           "km", "8.4"),
  }
}
(KOD / "ayar.json").write_text(json.dumps(ayar_json, ensure_ascii=False, indent=2), encoding="utf-8")

ayar_py = '''# Developed with assistance from Claude (Anthropic).
"""Orbitrary ayarları (belge 8.4): KOK, klasörler, tohum, parametre okuma, görev ağırlıkları."""
import json, hashlib
from pathlib import Path
import numpy as np

KOK = Path.home() / "orbitrary"
KLASOR = {ad: KOK / ad for ad in ("veri", "temiz", "kod", "ara", "sonuc", "docs", "makale", "kayit")}
AYAR_YOLU = KOK / "kod" / "ayar.json"
with open(AYAR_YOLU, encoding="utf-8") as _f:
    _AYAR = json.load(_f)

KATEGORILER = ["topografya", "topoloji", "jeoloji", "mineraloji", "termal", "su_buz", "toz", "aydinlanma"]  # k = 1..8
GOREVLER = ["genel", "mobility", "geology", "ice", "astro"]
CISIMLER = ["dunya", "ay", "mars"]


def _yaprak(anahtar):
    d = _AYAR["parametreler"]
    for p in anahtar.split("."):
        d = d[p]
    if not (isinstance(d, dict) and "deger" in d):
        raise KeyError(f"{anahtar} bir parametre değil")
    return d


def ayar(anahtar):
    """Noktalı anahtarın değeri (ör. 'kapi.buz.ana'). Belirlenmemiş değer okunursa ValueError."""
    d = _yaprak(anahtar)
    if d["deger"] is None:
        raise ValueError(f"{anahtar} henüz belirlenmedi: {d.get('durum')}")
    return d["deger"]


def kaynak(anahtar):
    return _yaprak(anahtar)["kaynak"]


def yapraklar(d=None, yol=""):
    """Bütün parametreler: (anahtar, yaprak sözlüğü)."""
    d = _AYAR["parametreler"] if d is None else d
    for k, v in d.items():
        y = f"{yol}.{k}" if yol else k
        if isinstance(v, dict) and "deger" in v:
            yield y, v
        elif isinstance(v, dict):
            yield from yapraklar(v, y)


TOHUM = ayar("tohum")


def rng(alt=None):
    """alt=None: default_rng(20261113); alt=sayı: aynı tohumdan sabit alt akış (ayar alt_tohum)."""
    if alt is None:
        return np.random.default_rng(TOHUM)
    return np.random.default_rng(np.random.SeedSequence(TOHUM, spawn_key=(int(alt),)))


def agirliklar(gorev, sema="kademeli"):
    """Belge 7.3: w_k(g); k = 1..8 sırasıyla 8 elemanlı dizi."""
    n = len(KATEGORILER)
    if gorev == "genel" or sema == "esit":
        return np.full(n, 1.0 / n)
    A = ayar("agirlik.ana_kategoriler")[gorev]
    m, pay = len(A), ayar("agirlik.ana_pay")
    return np.array([pay / m if k in A else (1 - pay) / (n - m) for k in range(1, n + 1)])


def ayar_sha256():
    return hashlib.sha256(AYAR_YOLU.read_bytes()).hexdigest()
'''
(KOD / "ayar.py").write_text(ayar_py, encoding="utf-8")

conftest = '''# Developed with assistance from Claude (Anthropic).
import sys
from pathlib import Path
sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
'''
(KOD / "testler" / "conftest.py").write_text(conftest, encoding="utf-8")

test_ayar = '''# Developed with assistance from Claude (Anthropic).
"""Belge 8.2 'Görev ağırlığı' testi + ayar.json bütünlük testleri."""
import numpy as np
import pytest
import ayar as A

# Belge 7.3 tablosu, belgeden aynen (k = 1..8)
TABLO_7_3 = {
    "genel":    [.125] * 8,
    "mobility": [.267, .267, .04, .04, .04, .04, .267, .04],
    "geology":  [.04, .267, .267, .267, .04, .04, .04, .04],
    "ice":      [.04, .04, .04, .04, .267, .267, .04, .267],
    "astro":    [.04, .04, .04, .267, .267, .267, .04, .04],
}


def test_klasorler_var():
    for ad, yol in A.KLASOR.items():
        assert yol.is_dir(), ad


def test_her_parametrede_kaynak():
    n = 0
    for anahtar, y in A.yapraklar():
        assert isinstance(y["kaynak"], str) and y["kaynak"].strip(), anahtar
        if y["deger"] is None:
            assert y.get("durum"), anahtar
        n += 1
    assert n >= 60


def test_belirlenmemis_deger_okunamaz():
    for anahtar in ("onele.K_on", "kapi.jeoloji.X"):
        with pytest.raises(ValueError):
            A.ayar(anahtar)


def test_gorev_agirligi_8_2():
    # m = 3, n = 8: 0,2667 × 3 + 0,04 × 5 = 1
    for g in ("mobility", "geology", "ice", "astro"):
        w = A.agirliklar(g)
        ana = np.isin(np.arange(1, 9), A.ayar("agirlik.ana_kategoriler")[g])
        assert ana.sum() == 3
        assert np.allclose(w[ana], 0.80 / 3) and round(w[ana][0], 4) == 0.2667
        assert np.allclose(w[~ana], 0.04)
        assert abs(w.sum() - 1) < 1e-12


def test_tablo_7_3_formulle_ayni():
    for g, beklenen in TABLO_7_3.items():
        assert np.array_equal(np.round(A.agirliklar(g), 3), np.array(beklenen)), g


def test_esit_sema():
    for g in A.GOREVLER:
        assert np.allclose(A.agirliklar(g, "esit"), 0.125)


def test_tohum_ve_tekrar():
    assert A.TOHUM == 20261113
    assert np.array_equal(A.rng().random(5), np.random.default_rng(20261113).random(5))
    assert np.array_equal(A.rng(5).random(5), A.rng(5).random(5))
    assert not np.array_equal(A.rng(2).random(5), A.rng(3).random(5))


def test_sabitler_4_1():
    assert A.ayar("cisim.yaricap_km") == {"dunya": 6371.0, "ay": 1737.4, "mars": 3389.5}
    assert A.ayar("cisim.eksen_egikligi_der") == {"dunya": 23.44, "ay": 1.54, "mars": 25.19}
    assert A.ayar("cisim.sigma_v_m") == {"dunya": 7.17, "ay": 10.0, "mars": 1.0}
    assert A.ayar("izgara.N") * A.ayar("izgara.delta_m") == A.ayar("izgara.L_km") * 1000


def test_analog_uzaklik_v121():
    d = A.ayar("analog_uzaklik_km")
    assert d["bilinen_alti"] == 20.0 and d["yakin_ust"] == 100.0
'''
(KOD / "testler" / "test_ayar.py").write_text(test_ayar, encoding="utf-8")

# Bölüm 11: değişiklik kaydı
deg = KAYIT / "degisiklik.md"
satir = ("- 2026-11-14 | σv Ay = 10 m, Mars = 1 m eklendi (belge 4.1'de değer yoktu) | "
         "kaynak [105] Smith ve ark. 2010 '~10 m radial', [104] Smith ve ark. 2001 '~1 m vertical' | "
         "etkisi: rölyef tabanı, κσv, teorem ℓ sabitleri; T2 parametre tablosu\n")
eski = deg.read_text(encoding="utf-8") if deg.exists() else "# Değişiklik kaydı (belge Bölüm 11)\n\n"
if "σv Ay = 10 m" not in eski:
    deg.write_text(eski + satir, encoding="utf-8")

# pytest
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "testler/test_ayar.py"],
                   cwd=KOD, capture_output=True, text=True)
cikti = r.stdout + r.stderr
(KAYIT / "D4_test_ayar.txt").write_text(cikti, encoding="utf-8")
print(cikti[-2500:])
zaman = datetime.datetime.now().isoformat(timespec="seconds")
durum = "PASS" if r.returncode == 0 else "FAIL"
with open(KAYIT / "14_kasim_log.txt", "a", encoding="utf-8") as f:
    f.write(f"{zaman}\tayar.py+test_ayar\t{durum}\n")
sys.path.insert(0, str(KOD))
import importlib, ayar as _A; importlib.reload(_A)
print(f"test_ayar: {durum} | parametre sayısı: {sum(1 for _ in _A.yapraklar())} | ayar.json SHA-256: {_A.ayar_sha256()[:16]}…")

.........                                                                [100%]
9 passed in 0.26s

test_ayar: PASS | parametre sayısı: 69 | ayar.json SHA-256: ab83b90bc08f7fe2…


In [4]:
# Developed with assistance from Claude (Anthropic).
# 17.4 sıra 2: kod\izgara.py + testler (belge 4.2–4.3, 8.3 "Lambert penceresi", "Halka ızgarası")
import json, sys, subprocess, datetime, importlib
from pathlib import Path
import numpy as np

KOK = Path.home() / "orbitrary"
KOD, KAYIT = KOK / "kod", KOK / "kayit"

izgara_py = '''# Developed with assistance from Claude (Anthropic).
"""Belge 4.2–4.3: halka ızgarası, tıklama eşlemesi, haversine komşular,
Lambert eş-alanlı azimutal dönüşüm (ileri/geri), pencere hücre merkezleri.
Açılar radyan, uzunluklar km. Pencere dizileri [j, i]: j kuzeye, i doğuya artar (belge 4.1: i doğu, j kuzey)."""
import numpy as np
from scipy.spatial import cKDTree
import ayar as A


def yaricap(govde):
    return float(A.ayar("cisim.yaricap_km")[govde])


def _yuvarla(x):
    # Yarımı yukarı yuvarlar; JavaScript Math.round ile aynı (site aynı formülü kullanır).
    return np.floor(np.asarray(x, dtype=float) + 0.5)


def haversine(phi1, lam1, phi2, lam2, r):
    """Belge 4.2 gc(P1, P2)."""
    s = np.sin((phi2 - phi1) / 2) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin((lam2 - lam1) / 2) ** 2
    return 2 * r * np.arcsin(np.sqrt(np.clip(s, 0.0, 1.0)))


def birim_vektor(phi, lam):
    c = np.cos(phi)
    return np.stack([c * np.cos(lam), c * np.sin(lam), np.sin(phi)], axis=-1)


class Halka:
    """Belge 4.2: M_B, Δφ, φ_i, m_i, λ_ij. Pencere kimliği = ofset[i] + j."""

    def __init__(self, govde):
        self.govde = govde
        self.r = yaricap(govde)
        D = float(A.ayar("izgara.Delta_km"))
        self.M = int(_yuvarla(np.pi * self.r / D))
        self.dphi = np.pi / self.M
        ih = np.arange(self.M)
        self.phi_halka = -np.pi / 2 + (ih + 0.5) * self.dphi
        self.m = np.maximum(1, _yuvarla(2 * np.pi * self.r * np.cos(self.phi_halka) / D)).astype(np.int64)
        self.ofset = np.concatenate([[0], np.cumsum(self.m)]).astype(np.int64)
        self.n = int(self.ofset[-1])
        self.i = np.repeat(ih, self.m)
        self.j = np.arange(self.n, dtype=np.int64) - self.ofset[self.i]
        self.phi = self.phi_halka[self.i]
        self.lam = (self.j + 0.5) * 2 * np.pi / self.m[self.i]
        self._agac = None

    def tikla(self, enlem_der, boylam_der):
        """Belge 4.2 tıklama eşlemesi (i*, j*) → pencere kimliği."""
        phi = np.radians(np.asarray(enlem_der, float))
        lam = np.radians(np.mod(np.asarray(boylam_der, float), 360.0))
        i = np.minimum(np.floor((phi + np.pi / 2) / self.dphi).astype(np.int64), self.M - 1)
        i = np.maximum(i, 0)
        j = np.minimum(np.floor(lam * self.m[i] / (2 * np.pi)).astype(np.int64), self.m[i] - 1)
        return self.ofset[i] + j

    def merkez_der(self, kimlik):
        k = np.asarray(kimlik)
        lon = np.degrees(self.lam[k])
        return np.degrees(self.phi[k]), np.where(lon > 180, lon - 360, lon)

    @property
    def agac(self):
        if self._agac is None:
            self._agac = cKDTree(birim_vektor(self.phi, self.lam))
        return self._agac

    def en_yakin(self, enlem_der, boylam_der, k=1):
        """KD-ağacıyla en yakın k merkez: (büyük çember km, kimlik)."""
        v = birim_vektor(np.radians(np.asarray(enlem_der, float)), np.radians(np.asarray(boylam_der, float)))
        d, idx = self.agac.query(v, k=k)
        return 2 * self.r * np.arcsin(np.clip(d / 2, 0, 1)), idx

    def komsular(self, enlem_der, boylam_der, n=None):
        """Belge 4.2: T'ye en yakın n pencere (T'nin kendisi hariç); uzaklık haversine ile."""
        n = int(A.ayar("izgara.n_komsu")) if n is None else int(n)
        _, idx = self.en_yakin(float(enlem_der), float(boylam_der), k=n + 1)
        p0, l0 = np.radians(float(enlem_der)), np.radians(float(boylam_der))
        d = haversine(p0, l0, self.phi[idx], self.lam[idx], self.r)
        tut = d > 1e-6
        idx, d = idx[tut][:n], d[tut][:n]
        assert len(idx) == n
        return idx, d


def lambert_ileri(phi, lam, phi0, lam0, r):
    """Belge 4.3 (Snyder [106]): (φ, λ) → (x, y), merkez (φ0, λ0)."""
    dl = lam - lam0
    den = 1 + np.sin(phi0) * np.sin(phi) + np.cos(phi0) * np.cos(phi) * np.cos(dl)
    k = np.sqrt(2 / np.maximum(den, 1e-300))
    x = r * k * np.cos(phi) * np.sin(dl)
    y = r * k * (np.cos(phi0) * np.sin(phi) - np.sin(phi0) * np.cos(phi) * np.cos(dl))
    return x, y


def lambert_geri(x, y, phi0, lam0, r):
    """Ters Lambert (Snyder [106]): (x, y) → (φ, λ ∈ [0, 2π))."""
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    rho = np.hypot(x, y)
    c = 2 * np.arcsin(np.clip(rho / (2 * r), -1, 1))
    sc, cc = np.sin(c), np.cos(c)
    with np.errstate(invalid="ignore", divide="ignore"):
        oran = np.where(rho > 0, y * sc * np.cos(phi0) / np.where(rho > 0, rho, 1.0), 0.0)
        phi = np.arcsin(np.clip(cc * np.sin(phi0) + oran, -1, 1))
        lam = lam0 + np.arctan2(x * sc, rho * np.cos(phi0) * cc - y * np.sin(phi0) * sc)
    phi = np.where(rho > 0, phi, phi0)
    lam = np.where(rho > 0, lam, lam0)
    return phi, np.mod(lam, 2 * np.pi)


def hucre_merkezleri(genisletme=0):
    """Belge 4.3: x_i = −L/2 + (i − ½)δ, i = 1−e … N+e (km)."""
    L = float(A.ayar("izgara.L_km"))
    d = float(A.ayar("izgara.delta_m")) / 1000.0
    N = int(A.ayar("izgara.N"))
    i = np.arange(1 - genisletme, N + genisletme + 1)
    return -L / 2 + (i - 0.5) * d


def pencere(phi0, lam0, r, genisletme=0):
    """Pencere hücre merkezlerinin (φ, λ) dizileri, biçim [j, i]."""
    c = hucre_merkezleri(genisletme)
    X, Y = np.meshgrid(c, c)
    return lambert_geri(X, Y, phi0, lam0, r)
'''
(KOD / "izgara.py").write_text(izgara_py, encoding="utf-8")

test_izgara = '''# Developed with assistance from Claude (Anthropic).
"""Belge 8.3: 'Lambert penceresi' ve 'Halka ızgarası' testleri (tanım: 14 Kasım notu, kayit\\\\degisiklik.md)."""
import numpy as np
import pytest
import ayar as A
import izgara as Z

G = ["dunya", "ay", "mars"]
D = 20.0


@pytest.fixture(scope="module")
def halkalar():
    return {g: Z.Halka(g) for g in G}


def test_haversine():
    r = 6371.0
    assert abs(Z.haversine(0.0, 0.0, 0.0, np.pi / 2, r) - np.pi * r / 2) < 1e-9
    assert abs(Z.haversine(np.pi / 2, 0.0, -np.pi / 2, 0.0, r) - np.pi * r) < 1e-6


@pytest.mark.parametrize("phi0_der", [0.0, 45.0, 89.9, -45.0, -89.9])
def test_lambert_gidis_donus(phi0_der):
    r, phi0, lam0 = 1737.4, np.radians(phi0_der), np.radians(179.9)
    g = np.random.default_rng(1)
    x, y = g.uniform(-15, 15, 5000), g.uniform(-15, 15, 5000)
    phi, lam = Z.lambert_geri(x, y, phi0, lam0, r)
    x2, y2 = Z.lambert_ileri(phi, lam, phi0, lam0, r)
    assert np.max(np.abs(x2 - x)) < 1e-9 and np.max(np.abs(y2 - y)) < 1e-9


def _kure_alan(phi, lam, merkez, r):
    # Merkezden yelpaze üçgenler; tan(E/2) = |c·(a×b)| / (1 + a·b + b·c + c·a)
    a = Z.birim_vektor(phi, lam)
    b = np.roll(a, -1, axis=0)
    c = Z.birim_vektor(*merkez)
    ust = np.abs(np.cross(a, b) @ c)
    alt = 1 + np.einsum("ij,ij->i", a, b) + a @ c + b @ c
    return np.sum(2 * np.arctan2(ust, alt)) * r * r


@pytest.mark.parametrize("govde", G)
@pytest.mark.parametrize("phi0_der", [0.0, 45.0, 89.9])
def test_lambert_pencere_alani(govde, phi0_der):
    r, L = Z.yaricap(govde), float(A.ayar("izgara.L_km"))
    h, k = L / 2, 2000
    t = np.linspace(-h, h, k, endpoint=False)
    x = np.concatenate([t, np.full(k, h), -t, np.full(k, -h)])
    y = np.concatenate([np.full(k, -h), t, np.full(k, h), -t])
    phi0, lam0 = np.radians(phi0_der), np.radians(179.9)
    phi, lam = Z.lambert_geri(x, y, phi0, lam0, r)
    alan = _kure_alan(phi, lam, (phi0, lam0), r)
    assert abs(alan / (L * L) - 1) < 1e-3, alan


def test_pencere_merkezleri():
    N, r = int(A.ayar("izgara.N")), 6371.0
    phi0, lam0 = np.radians(45.0), np.radians(179.9)
    c = Z.hucre_merkezleri()
    assert len(c) == N and abs(c[0] + 9.75) < 1e-12 and abs(c[-1] - 9.75) < 1e-12
    phi, lam = Z.pencere(phi0, lam0, r)
    assert phi.shape == (N, N)
    x, y = Z.lambert_ileri(phi, lam, phi0, lam0, r)
    assert np.allclose(x, c[None, :], atol=1e-9) and np.allclose(y, c[:, None], atol=1e-9)
    assert np.all(np.diff(phi[:, N // 2]) > 0)
    assert Z.pencere(phi0, lam0, r, genisletme=2)[0].shape == (N + 4, N + 4)


@pytest.mark.parametrize("govde", G)
def test_halka_araliklari(halkalar, govde):
    H = halkalar[govde]
    assert abs(H.r * H.dphi / D - 1) < 0.05
    s = 2 * np.pi * H.r * np.cos(H.phi_halka) / H.m
    assert np.all(np.abs(s / D - 1) < 0.05), float(s[np.argmax(np.abs(s / D - 1))])
    assert abs(H.n / (4 * np.pi * H.r ** 2 / D ** 2) - 1) < 0.01
    d, _ = H.agac.query(Z.birim_vektor(H.phi, H.lam), k=2)
    gc = 2 * H.r * np.arcsin(d[:, 1] / 2)
    ic = (H.i > 0) & (H.i < H.M - 1)
    assert gc[ic].min() > 0.95 * D and gc[ic].max() < 1.05 * D, (gc[ic].min(), gc[ic].max())
    assert H.m[0] == H.m[-1] == 3


def _kure_noktalari(g, n):
    lat = np.degrees(np.arcsin(g.uniform(-1, 1, n)))
    lon = g.uniform(-180, 180, n)
    zc = g.uniform(np.sin(np.radians(85)), 1, n)
    isaret = g.choice([-1.0, 1.0], n)
    lat = np.concatenate([lat, isaret * np.degrees(np.arcsin(zc)), [90.0, -90.0]])
    lon = np.concatenate([lon, g.uniform(-180, 180, n), [0.0, 0.0]])
    return lat, lon


@pytest.mark.parametrize("govde", G)
def test_kutupta_bosluk_yok(halkalar, govde):
    H = halkalar[govde]
    lat, lon = _kure_noktalari(np.random.default_rng(2), 100000)
    d, _ = H.en_yakin(lat, lon)
    assert d.max() < 15.0, d.max()


@pytest.mark.parametrize("govde", G)
def test_tiklama_merkez_ve_yakini(halkalar, govde):
    H = halkalar[govde]
    g = np.random.default_rng(3)
    k = np.unique(np.concatenate([g.choice(H.n, 20000, replace=False), H.ofset[:-1], H.ofset[1:] - 1]))
    lat, lon = H.merkez_der(k)
    assert np.array_equal(H.tikla(lat, lon), k)
    rr, az = g.uniform(0, 5, len(k)), g.uniform(0, 2 * np.pi, len(k))
    phi, lam = Z.lambert_geri(rr * np.sin(az), rr * np.cos(az), H.phi[k], H.lam[k], H.r)
    plat, plon = np.degrees(phi), np.degrees(lam)
    assert np.array_equal(H.tikla(plat, plon), k)
    _, yak = H.en_yakin(plat, plon)
    assert np.array_equal(yak, k)


@pytest.mark.parametrize("govde", G)
def test_tiklama_rastgele(halkalar, govde):
    H = halkalar[govde]
    lat, lon = _kure_noktalari(np.random.default_rng(4), 100000)
    k = H.tikla(lat, lon)
    d = Z.haversine(np.radians(lat), np.radians(lon), H.phi[k], H.lam[k], H.r)
    assert d.max() < 15.0, d.max()


def test_komsular(halkalar):
    H = halkalar["ay"]
    lat, lon = H.merkez_der(12345)
    idx, d = H.komsular(lat, lon)
    assert len(idx) == 20 and 12345 not in idx and np.all(np.diff(d) >= -1e-9) and d.max() < 60
    idx2, d2 = H.komsular(-89.0, 30.0)
    assert len(idx2) == 20 and d2.max() < 60
'''
(KOD / "testler" / "test_izgara.py").write_text(test_izgara, encoding="utf-8")

# FONKSIYONLAR.md (bölüm bölüm; her modül kendi bölümünü yeniler)
def fonksiyon_bolumu(ad, satirlar):
    yol = KOD / "FONKSIYONLAR.md"
    metin = yol.read_text(encoding="utf-8") if yol.exists() else "# Denklem → fonksiyon (Emir'in dipnotları için)\n"
    bas, son = f"<!-- {ad} -->", f"<!-- /{ad} -->"
    blok = f"{bas}\n## {ad}\n" + "\n".join(f"- {s}" for s in satirlar) + f"\n{son}"
    if bas in metin:
        metin = metin.split(bas)[0] + blok + metin.split(son)[1]
    else:
        metin = metin.rstrip() + "\n\n" + blok + "\n"
    yol.write_text(metin, encoding="utf-8")

fonksiyon_bolumu("ayar.py", ["7.3 w_k(g) → `ayar.agirliklar(gorev, sema)`",
                             "8.4 parametreler → `ayar.ayar(anahtar)`, kaynak → `ayar.kaynak(anahtar)`",
                             "17 rastgelelik → `ayar.rng(alt)`"])
fonksiyon_bolumu("izgara.py", ["4.2 M_B, Δφ, φ_i, m_i, λ_ij → `izgara.Halka(govde)`",
                               "4.2 (i*, j*) tıklama eşlemesi → `izgara.Halka.tikla`",
                               "4.2 gc(P1, P2) → `izgara.haversine`; komşular N(T) → `izgara.Halka.komsular`",
                               "4.3 Lambert k′, x, y → `izgara.lambert_ileri`; ters → `izgara.lambert_geri`",
                               "4.3 x_i, y_j → `izgara.hucre_merkezleri`, `izgara.pencere`"])

# Değişiklik kaydı
deg = KAYIT / "degisiklik.md"
satir = ("- 2026-11-14 | 8.3 'Halka ızgarası' testi kodda daraltıldı: formül aralıkları her halkada 20 km ± %5; "
         "gerçek komşu uzaklığı kutup ilk halkası (m = 3) dışında ± %5; ≤ 5 km sapan tıklama kendi penceresi = en yakın merkez; "
         "her nokta ≤ 15 km merkeze eşlenir ('kutupta boşluk yok' aynı sınırla) | neden: kutup halkasında 3 pencere arası ~17,3 km; "
         "halka sınırında en yakın merkez komşu halkada olabilir | etkisi: T5\n")
eski = deg.read_text(encoding="utf-8")
if "Halka ızgarası' testi kodda daraltıldı" not in eski:
    deg.write_text(eski + satir, encoding="utf-8")

# pytest (bütün testler)
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "testler"],
                   cwd=KOD, capture_output=True, text=True)
cikti = r.stdout + r.stderr
(KAYIT / "D4_test_izgara.txt").write_text(cikti, encoding="utf-8")
print(cikti[-3000:])
durum = "PASS" if r.returncode == 0 else "FAIL"

# Bilgi (makale için sayılar kayda; elle yazılmaz)
if str(KOD) not in sys.path:
    sys.path.insert(0, str(KOD))
import ayar, izgara
importlib.reload(ayar); importlib.reload(izgara)
bilgi = {}
g = np.random.default_rng(9)
for gv in ["dunya", "ay", "mars"]:
    H = izgara.Halka(gv)
    lat = np.degrees(np.arcsin(g.uniform(-1, 1, 200000))); lon = g.uniform(-180, 180, 200000)
    k = H.tikla(lat, lon); _, yak = H.en_yakin(lat, lon)
    d0, _ = H.agac.query(izgara.birim_vektor(H.phi[:3], H.lam[:3]), k=2)
    bilgi[gv] = {"pencere_sayisi": H.n, "halka_sayisi": H.M,
                 "tiklama_en_yakin_orani": round(float(np.mean(k == yak)), 4),
                 "kutup_ilk_halka_m": int(H.m[0]),
                 "kutup_ilk_halka_komsu_km": round(float(2 * H.r * np.arcsin(d0[0, 1] / 2)), 2)}
(KAYIT / "izgara_bilgi.json").write_text(json.dumps(bilgi, ensure_ascii=False, indent=2), encoding="utf-8")
with open(KAYIT / "14_kasim_log.txt", "a", encoding="utf-8") as f:
    f.write(f"{datetime.datetime.now().isoformat(timespec='seconds')}\tizgara.py+test_izgara\t{durum}\n")
print(f"\ntestler: {durum}")
for gv, b in bilgi.items():
    print(f"{gv:6s} pencere {b['pencere_sayisi']:>9,d} | halka {b['halka_sayisi']:5d} | "
          f"tıklama=en yakın %{100*b['tiklama_en_yakin_orani']:.2f} | kutup ilk halka m={b['kutup_ilk_halka_m']}, komşu {b['kutup_ilk_halka_komsu_km']} km")

est_tiklama_rastgele(halkalar, govde):
        H = halkalar[govde]
        lat, lon = _kure_noktalari(np.random.default_rng(4), 100000)
        k = H.tikla(lat, lon)
        d = Z.haversine(np.radians(lat), np.radians(lon), H.phi[k], H.lam[k], H.r)
>       assert d.max() < 15.0, d.max()
E       AssertionError: np.float64(17.025177148029076)
E       assert np.float64(17.025177148029076) < 15.0
E        +  where np.float64(17.025177148029076) = <built-in method max of numpy.ndarray object at 0x00000230E651CC30>()
E        +    where <built-in method max of numpy.ndarray object at 0x00000230E651CC30> = array([11.33642954,  6.6671167 ,  8.07316347, ...,  9.01072443,\n        9.99670893,  9.99670893], shape=(200002,)).max

testler\test_izgara.py:123: AssertionError
_________________________ test_tiklama_rastgele[mars] _________________________

halkalar = {'dunya': <izgara.Halka object at 0x00000230E3540620>, 'ay': <izgara.Halka object at 0x00000230E3540650>, 'mars': <izgara.Halka object at

In [5]:
# Developed with assistance from Claude (Anthropic).
# izgara testi düzeltmesi: test_tiklama_rastgele → hücre içinde + köşe uzaklığı sınırı (ilk FAIL kayıtta kalır)
import json, sys, subprocess, datetime, importlib
from pathlib import Path
import numpy as np

KOK = Path.home() / "orbitrary"
KOD, KAYIT = KOK / "kod", KOK / "kayit"
yol = KOD / "testler" / "test_izgara.py"
metin = yol.read_text(encoding="utf-8")

eski = '''@pytest.mark.parametrize("govde", G)
def test_tiklama_rastgele(halkalar, govde):
    H = halkalar[govde]
    lat, lon = _kure_noktalari(np.random.default_rng(4), 100000)
    k = H.tikla(lat, lon)
    d = Z.haversine(np.radians(lat), np.radians(lon), H.phi[k], H.lam[k], H.r)
    assert d.max() < 15.0, d.max()
'''
yeni = '''def kose_uzakligi(H):
    """Her halkada hücre köşesinin merkeze büyük çember uzaklığı (hücrenin en uzak noktası)."""
    ust = np.minimum(H.phi_halka + H.dphi / 2, np.pi / 2)
    alt = np.maximum(H.phi_halka - H.dphi / 2, -np.pi / 2)
    yarim = np.pi / H.m
    return np.maximum(Z.haversine(H.phi_halka, 0.0, ust, yarim, H.r),
                      Z.haversine(H.phi_halka, 0.0, alt, yarim, H.r))


@pytest.mark.parametrize("govde", G)
def test_tiklama_rastgele(halkalar, govde):
    H = halkalar[govde]
    lat, lon = _kure_noktalari(np.random.default_rng(4), 100000)
    k = H.tikla(lat, lon)
    phi, lam = np.radians(lat), np.radians(np.mod(lon, 360.0))
    i, j = H.i[k], H.j[k]
    e = 1e-12
    assert np.all(phi >= H.phi_halka[i] - H.dphi / 2 - e) and np.all(phi <= H.phi_halka[i] + H.dphi / 2 + e)
    assert np.all(lam >= j * 2 * np.pi / H.m[i] - e) and np.all(lam < (j + 1) * 2 * np.pi / H.m[i] + e)
    d = Z.haversine(phi, lam, H.phi[k], H.lam[k], H.r)
    asim = d - kose_uzakligi(H)[i]
    assert asim.max() <= 1e-6, float(asim.max())
'''
assert metin.count(eski) == 1, "eski test fonksiyonu dosyada tam bir kez bulunamadı"
yol.write_text(metin.replace(eski, yeni), encoding="utf-8")

deg = KAYIT / "degisiklik.md"
satir = ("- 2026-11-14 | test_tiklama_rastgele düzeltildi: sabit 15 km sınırı yerine 'nokta kendi hücresinde ve merkeze uzaklık "
         "≤ hücre köşe uzaklığı' | neden: kutup ilk iki halkasında hücre dilim biçimli (m = 3, 9), köşe 15 km'yi aşar; "
         "ilk koşu FAIL (16,5 / 17,0 / 16,9 km) kayit\\D4_test_izgara.txt'de | etkisi: T5\n")
d_eski = deg.read_text(encoding="utf-8")
if "test_tiklama_rastgele düzeltildi" not in d_eski:
    deg.write_text(d_eski + satir, encoding="utf-8")

r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "testler"],
                   cwd=KOD, capture_output=True, text=True)
cikti = r.stdout + r.stderr
(KAYIT / "D4_test_izgara_2.txt").write_text(cikti, encoding="utf-8")
print(cikti[-2000:])
durum = "PASS" if r.returncode == 0 else "FAIL"

if str(KOD) not in sys.path:
    sys.path.insert(0, str(KOD))
import ayar, izgara
importlib.reload(ayar); importlib.reload(izgara)
sys.path.insert(0, str(KOD / "testler"))
import test_izgara; importlib.reload(test_izgara)

bilgi = json.loads((KAYIT / "izgara_bilgi.json").read_text(encoding="utf-8"))
for gv in ["dunya", "ay", "mars"]:
    H = izgara.Halka(gv)
    ku = test_izgara.kose_uzakligi(H)
    bilgi[gv]["kose_km"] = {"halka_0": round(float(ku[0]), 2), "halka_1": round(float(ku[1]), 2),
                            "diger_en_buyuk": round(float(ku[2:H.M - 2].max()), 2)}
(KAYIT / "izgara_bilgi.json").write_text(json.dumps(bilgi, ensure_ascii=False, indent=2), encoding="utf-8")
with open(KAYIT / "14_kasim_log.txt", "a", encoding="utf-8") as f:
    f.write(f"{datetime.datetime.now().isoformat(timespec='seconds')}\ttest_izgara (2. koşu, köşe sınırı)\t{durum}\n")
print(f"\ntestler: {durum}")
for gv, b in bilgi.items():
    print(f"{gv:6s} köşe uzaklığı: halka0 {b['kose_km']['halka_0']} km | halka1 {b['kose_km']['halka_1']} km | diğer en büyük {b['kose_km']['diger_en_buyuk']} km")

......................................                                   [100%]
38 passed in 2.83s


testler: PASS
dunya  köşe uzaklığı: halka0 17.32 km | halka1 15.64 km | diğer en büyük 14.67 km
ay     köşe uzaklığı: halka0 17.31 km | halka1 15.64 km | diğer en büyük 14.67 km
mars   köşe uzaklığı: halka0 17.33 km | halka1 15.66 km | diğer en büyük 14.68 km


In [6]:
# Developed with assistance from Claude (Anthropic).
# katman.py öncesi: temiz\ envanteri (yalnız başlık/şekil/yan .json; veri okunmaz) + makine belleği
import json, re, os, ctypes, datetime
from pathlib import Path
from collections import defaultdict
import numpy as np
import rasterio, pyogrio

KOK = Path.home() / "orbitrary"
TEMIZ, KAYIT = KOK / "temiz", KOK / "kayit"

def kalip(ad):  # tarih/sayı/karo adlarını tek kalıba indir
    return re.sub(r"\d", "#", ad)

def kisa(o, n=300):
    s = json.dumps(o, ensure_ascii=False, default=str)
    return s if len(s) <= n else s[:n] + "…"

envanter, satirlar = {}, []
for govde in ["dunya", "ay", "mars"]:
    for kl in sorted(p for p in (TEMIZ / govde).iterdir() if p.is_dir()):
        ad = f"{govde}/{kl.name}"
        meta = json.loads((kl / "meta.json").read_text(encoding="utf-8"))
        gruplar = defaultdict(list)
        for p in sorted(kl.rglob("*")):
            if p.is_file() and p.name != "meta.json":
                gruplar[kalip(p.relative_to(kl).as_posix())].append(p)
        kayit = {"meta_anahtarlar": list(meta.keys()), "dosya_gruplari": {}}
        satirlar.append(f"\n### {ad}  | meta anahtarları: {list(meta.keys())}")
        for k, liste in gruplar.items():
            p = liste[0]; bilgi = {"adet": len(liste), "ornek": p.name, "bayt_ornek": p.stat().st_size}
            try:
                if p.suffix == ".npy":
                    a = np.load(p, mmap_mode="r"); bilgi.update(shape=list(a.shape), dtype=str(a.dtype))
                elif p.suffix == ".json":
                    bilgi["icerik"] = json.loads(p.read_text(encoding="utf-8"))
                elif p.suffix.lower() in (".tif", ".tiff"):
                    with rasterio.open(p) as ds:
                        bilgi.update(boyut=[ds.height, ds.width], bant=ds.count, dtype=ds.dtypes[0], nodata=ds.nodata,
                                     transform=list(ds.transform)[:6], crs=(ds.crs.to_string()[:80] if ds.crs else None),
                                     olcek=list(ds.scales), ofset=list(ds.offsets), blok=ds.block_shapes[0])
                elif p.suffix == ".gpkg":
                    inf = pyogrio.read_info(p)
                    bilgi.update(katman=[l[0] for l in pyogrio.list_layers(p)], kayit=inf["features"],
                                 sutunlar=list(inf["fields"]), crs=str(inf["crs"])[:80], geometri=inf["geometry_type"])
            except Exception as e:
                bilgi["hata"] = repr(e)[:200]
            kayit["dosya_gruplari"][k] = bilgi
            goster = {x: bilgi[x] for x in bilgi if x not in ("ornek", "bayt_ornek")}
            satirlar.append(f"  {k}  ×{bilgi['adet']}  {kisa(goster, 420)}")
        # meta'dan okuma için kritik alanlar (adı ne olursa olsun, 'crs', 'olcek', 'veri', 'cozun', 'konum' geçenler)
        kritik = {x: meta[x] for x in meta if re.search(r"crs|olcek|ofset|veri|cozun|konum|birim|izgara|geotrans", x, re.I)}
        kayit["meta_kritik"] = kritik
        satirlar.append(f"  meta(kritik): {kisa(kritik, 500)}")
        envanter[ad] = kayit

# makine
class MS(ctypes.Structure):
    _fields_ = [("dwLength", ctypes.c_ulong), ("dwMemoryLoad", ctypes.c_ulong), ("ullTotalPhys", ctypes.c_ulonglong),
                ("ullAvailPhys", ctypes.c_ulonglong), ("ullTotalPageFile", ctypes.c_ulonglong), ("ullAvailPageFile", ctypes.c_ulonglong),
                ("ullTotalVirtual", ctypes.c_ulonglong), ("ullAvailVirtual", ctypes.c_ulonglong), ("ullAvailExtendedVirtual", ctypes.c_ulonglong)]
ms = MS(); ms.dwLength = ctypes.sizeof(MS); ctypes.windll.kernel32.GlobalMemoryStatusEx(ctypes.byref(ms))
import shutil
makine = {"cpu": os.cpu_count(), "ram_gb": round(ms.ullTotalPhys / 1e9, 1), "bos_ram_gb": round(ms.ullAvailPhys / 1e9, 1),
          "bos_disk_gb": round(shutil.disk_usage(KOK).free / 1e9, 1)}
envanter["_makine"] = makine
(KAYIT / "temiz_envanter.json").write_text(json.dumps(envanter, ensure_ascii=False, indent=1, default=str), encoding="utf-8")
with open(KAYIT / "14_kasim_log.txt", "a", encoding="utf-8") as f:
    f.write(f"{datetime.datetime.now().isoformat(timespec='seconds')}\ttemiz envanteri\tOK\n")
print(f"MAKİNE: {makine}")
print("\n".join(satirlar))
print(f"\n→ kayit\\temiz_envanter.json ({len(envanter)-1} set)")

MAKİNE: {'cpu': 12, 'ram_gb': 34.1, 'bos_ram_gb': 17.1, 'bos_disk_gb': 108.2}

### dunya/E10_soilgrids  | meta anahtarları: ['set', 'ad', 'doi', 'surum', 'lisans', 'birim', 'kaynak_adresleri', 'indirme_tarihi', 'ham_sha256', 'gercek_yer_testleri_D2', 'D3_testleri', 'crs', 'piksel', 'olcek', 'ofset', 'veri_yok_kurali', 'okuma', 'dosyalar', 'uretim_zamani', 'uretim_kodu']
  kil_#-#cm.tif  ×1  {"adet": 1, "boyut": [14509, 39812], "bant": 1, "dtype": "float32", "nodata": NaN, "transform": [1000.0, 0.0, -19949750.0, 0.0, -1000.0, 8361000.000000001], "crs": "ESRI:54052", "olcek": [1.0], "ofset": [0.0], "blok": [512, 512]}
  silt_#-#cm.tif  ×1  {"adet": 1, "boyut": [14509, 39812], "bant": 1, "dtype": "float32", "nodata": NaN, "transform": [1000.0, 0.0, -19949750.0, 0.0, -1000.0, 8361000.000000001], "crs": "ESRI:54052", "olcek": [1.0], "ofset": [0.0], "blok": [512, 512]}
  meta(kritik): {"birim": "‰ (g/kg)", "crs": "ESRI:54052", "olcek": 1.0, "ofset": 0.0, "veri_yok_kurali": "ham nodata (dosya

In [7]:
   %run C:/Users/USER/orbitrary/kod/hucre_katman.py

________ test_pgda_en_soguk_ucgenler_guney ______________________

>   self._hds = open_dataset(

rasterio/_base.pyx:322: 
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _
rasterio/_base.pyx:234: in rasterio._base.open_dataset
    return exc_wrap_pointer(hds)
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _

>   ???
E   rasterio._err.CPLE_OpenFailedError: C:\Users\USER\orbitrary\temiz\ay\M7_pgda\lpsr_s.tif: No such file or directory

rasterio/_err.pyx:359: CPLE_OpenFailedError

During handling of the above exception, another exception occurred:

    def test_pgda_en_soguk_ucgenler_guney():
        \x1b# D2 testinin katman.py ile tekrar\u0131: |enlem| \u2265 85\xb0 g\xfcney en so\u011fuk 100 PRP \xfc\xe7geni (temp_max) \u2192 LPSR = 1 oran\u0131 \u2265 0,9\x1b\x1b\x1b
        kl = K._kl("ay", "M4")
        lat = np.load(kl / "prp_south_tri_clat.npy"); lon = np.load(kl / "prp_south_tri_clon.npy")
        t = np.load(kl / "prp_

In [8]:
# Developed with assistance from Claude (Anthropic).
# katman.py düzeltmesi: PGDA dosya adı (avgvisib_65n.tif …) + çıkışta rasterio dosyalarını kapat. İlk FAIL kayıtta kalır.
import sys, subprocess, datetime
from pathlib import Path

KOK = Path.home() / "orbitrary"
KOD, KAYIT = KOK / "kod", KOK / "kayit"
yol = KOD / "katman.py"
s = yol.read_text(encoding="utf-8")

eski1 = '''        kutup = ad[-1]
        ds = rasterio.open(_kl("ay", "M7") / f"{ad}.tif")'''
yeni1 = '''        kutup = ad[-1]
        dosya = _kl("ay", "M7") / f"{ad[:-2]}_65{kutup}.tif"   # temiz\\ay\\M7_pgda: avgvisib_65n.tif, lpsr_65s.tif …
        assert dosya.is_file(), dosya
        ds = rasterio.open(dosya)'''
eski2 = "_ONBELLEK = {}\n"
yeni2 = '''_ONBELLEK = {}


def _kapat():
    """Çıkışta açık rasterio dosyalarını kapat (yorumlayıcı kapanırken excepthook uyarısını önler)."""
    for k in list(_ONBELLEK.values()):
        ds = getattr(k, "kaynak", None)
        if hasattr(ds, "close"):
            try:
                ds.close()
            except Exception:
                pass


import atexit
atexit.register(_kapat)
'''
for e in (eski1, eski2):
    assert s.count(e) == 1, f"değiştirilecek metin {s.count(e)} kez bulundu:\n{e}"
s = s.replace(eski1, yeni1).replace(eski2, yeni2)
yol.write_text(s, encoding="utf-8")

deg = KAYIT / "degisiklik.md"
satir = ("- 2026-11-14 | katman.py: PGDA dosya adı düzeltildi (avgvisib_65n/65s, lpsr_65n/65s) | ilk koşu 6 FAIL "
         "(dosya bulunamadı), kayit\\D4_test_katman.txt | etkisi: yok (kod)\n")
d = deg.read_text(encoding="utf-8")
if "PGDA dosya adı düzeltildi" not in d:
    deg.write_text(d + satir, encoding="utf-8")

r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "testler"],
                   cwd=KOD, capture_output=True, text=True)
cikti = r.stdout + r.stderr
(KAYIT / "D4_test_katman_2.txt").write_text(cikti, encoding="utf-8")
print(cikti[-3500:])
durum = "PASS" if r.returncode == 0 else "FAIL"
with open(KAYIT / "14_kasim_log.txt", "a", encoding="utf-8") as f:
    f.write(f"{datetime.datetime.now().isoformat(timespec='seconds')}\ttest_katman 2. koşu (PGDA adı)\t{durum}\n")
print(f"\ntestler: {durum}")

........................................................................ [ 60%]
................................................                         [100%]
120 passed in 133.39s (0:02:13)


testler: PASS


In [9]:
%run C:/Users/USER/orbitrary/kod/hucre_uzaklik.py

........................................................................ [ 50%]
........................................................................ [100%]
144 passed in 133.18s (0:02:13)


testler: PASS


In [10]:
%run C:/Users/USER/orbitrary/kod/hucre_ozellik_topo.py

........................................................................ [ 44%]
........................................................................ [ 89%]
.................                                                        [100%]
161 passed in 138.85s (0:02:18)


testler: PASS


In [11]:
%run C:/Users/USER/orbitrary/kod/hucre_ozellik_topoloji.py

.................................................. [ 38%]
........................................................................ [ 57%]
........................................................................ [ 77%]
........................................................................ [ 96%]
.............                                                            [100%]
373 passed in 141.16s (0:02:21)


testler: PASS


In [12]:
# Developed with assistance from Claude (Anthropic).
# Girdi denetimi: Sude'nin dosyaları (belge 2.3, 15.1) var mı, belgeye uyuyor mu. Hiçbir dosyayı değiştirmez.
import json, datetime
from pathlib import Path
import pandas as pd, pyogrio

KOK = Path.home() / "orbitrary"; KAYIT = KOK / "kayit"; TEMIZ = KOK / "temiz"
ARANAN = {  # belge 2.3 / 15.1 / 17.6
    "hedefler.csv": ["kimlik", "ad", "cisim", "enlem", "boylam", "boylam_yonu", "kaynak", "kaynak_yeri", "not"],
    "jeoloji_sozluk.csv": ["cisim", "birim_kodu", "birim_aciklama", "grup", "kaynak"],
    "dogrulama_ciftleri.csv": None, "v10_lambert_reimold.csv": None, "analog_metin.csv": ["cift_no", "baslik", "metin", "kaynak_url"],
    "metin.json": None}
yerler = [KOK] + [Path.home() / d for d in ("Downloads", "Desktop", "Documents", "İndirilenler", "Masaüstü", "Belgeler")]
atla = {"veri", "temiz", "ara", ".git"}
bulunan = {ad: [] for ad in ARANAN}
for kok in yerler:
    if not kok.is_dir(): continue
    for ad in ARANAN:
        for p in kok.rglob(ad):
            if not (set(p.relative_to(kok).parts) & atla):
                bulunan[ad].append(p)

rapor = {"zaman": datetime.datetime.now().isoformat(timespec="seconds"), "dosyalar": {}}
def oku(p):
    for enc in ("utf-8-sig", "cp1254", "latin-1"):
        try: return pd.read_csv(p, encoding=enc, dtype=str), enc
        except Exception: pass
    return None, None

print("=== Dosyalar ===")
for ad, beklenen in ARANAN.items():
    yollar = sorted(set(bulunan[ad]))
    r = {"yollar": [str(p) for p in yollar]}
    if not yollar:
        print(f"YOK   {ad}")
    for p in yollar:
        print(f"VAR   {ad}  →  {p}")
        if p.suffix == ".csv":
            df, enc = oku(p)
            if df is None: print("      okunamadı"); continue
            r.update(satir=len(df), sutunlar=list(df.columns), kodlama=enc)
            print(f"      {len(df)} satır | sütunlar: {list(df.columns)}")
            if beklenen:
                eksik = [c for c in beklenen if c not in df.columns]
                r["eksik_sutun"] = eksik
                print(f"      belge 15.1 sütunlarından eksik: {eksik or 'yok'}")
    rapor["dosyalar"][ad] = r

# --- hedefler.csv içerik kontrolü (belge 9.1–9.2 + indirme_raporu3 B5)
BEKLENEN_HEDEF = {  # belgede ya da B5'te sayısı yazılı olanlar
    "A3-01": (-83.576, -69.076), "A3-02": (-86.799, -22.898), "A3-03": (-86.047, 0.149), "A3-04": (-84.534, 30.655),
    "A3-05": (-85.399, 31.180), "A3-06": (-85.188, 37.240), "A3-07": (-83.888, 58.403), "A3-08": (-88.174, -68.787),
    "A3-09": (-87.207, 126.606), "M-SP": (-89.0, 30.0), "M-A14": (-3.64589, -17.47194), "M-A15": (26.13239, 3.63330),
    "M-A16": (-8.9734, 15.5011), "M-A17": (20.1911, 30.7723), "R-JEZ": (18.44, 77.45), "R-GAL": (-4.5895, 137.4417),
    "R-PF": (19.13, -33.22)}
SUDE_DOLDURUR = ["M-A11", "M-SHK", "R-GUS", "R-MER", "R-PHX", "R-UTO", "R-ELY", "R-OLY"]  # belge 9.1–9.2
print("\n=== hedefler.csv içeriği ===")
if bulunan["hedefler.csv"]:
    df, _ = oku(sorted(bulunan["hedefler.csv"])[0])
    if "kimlik" in df.columns:
        df = df.set_index("kimlik")
        for k in list(BEKLENEN_HEDEF) + SUDE_DOLDURUR:
            if k not in df.index: print(f"  {k:6s} YOK"); continue
            try: la, lo = float(str(df.loc[k, "enlem"]).replace(",", ".")), float(str(df.loc[k, "boylam"]).replace(",", "."))
            except Exception: print(f"  {k:6s} koordinat boş/okunamadı"); continue
            if k in BEKLENEN_HEDEF:
                e = BEKLENEN_HEDEF[k]; ok = abs(la - e[0]) < 1e-3 and abs(lo - e[1]) < 1e-3
                print(f"  {k:6s} {la:.5f}, {lo:.5f}  {'= belge/B5' if ok else f'≠ belge/B5 {e}'}")
            else:
                print(f"  {k:6s} {la:.5f}, {lo:.5f}  (kaynak: {df.loc[k].get('kaynak', '?')})")
else:
    print("  dosya yok")

# --- jeoloji birimleri: verideki gerçek liste (Sude'nin sözlüğü bununla eşleşmeli)
def birimler(gpkg, katman):
    t = pyogrio.read_dataframe(gpkg, layer=katman, columns=["birim", "alan_km2"], read_geometry=False)
    return t.groupby("birim")["alan_km2"].sum().sort_values(ascending=False)
gl = birimler(next((TEMIZ/"dunya").glob("E3_*"))/"glim.gpkg", "glim")
ay = birimler(next((TEMIZ/"ay").glob("M2_*"))/"ay_jeoloji.gpkg", "ay_jeoloji")
ma = birimler(next((TEMIZ/"mars").glob("R2_*"))/"sim3292.gpkg", "sim3292")
print(f"\n=== Verideki jeoloji birimleri === GLiM {len(gl)} sınıf: {list(gl.index)}")
print(f"Ay {len(ay)} birim; belge 6.3 mare listesi verideki: {[b for b in ['Im1','Im2','Em','Imd','Id','EIp'] if b in ay.index]}, verideki değil: {[b for b in ['Im1','Im2','Em','Imd','Id','EIp'] if b not in ay.index]}")
print(f"Mars {len(ma)} birim: {list(ma.index)}")
sablon = pd.concat([pd.DataFrame({"cisim": c, "birim_kodu": s.index, "alan_km2_toplam": s.values.round(1)})
                    for c, s in (("dunya", gl), ("ay", ay), ("mars", ma))])
for c in ("birim_aciklama", "grup", "kaynak"): sablon[c] = ""
sablon.to_csv(KAYIT / "sude_jeoloji_sablon.csv", index=False, encoding="utf-8-sig")

if bulunan["jeoloji_sozluk.csv"]:
    sz, _ = oku(sorted(bulunan["jeoloji_sozluk.csv"])[0])
    if {"cisim", "birim_kodu", "grup"} <= set(sz.columns):
        for c, s in (("mars", ma), ("ay", ay), ("dunya", gl)):
            kodlar = set(sz.loc[sz["cisim"].str.lower().str.strip() == c, "birim_kodu"].str.strip())
            print(f"  sözlük {c}: verideki {len(s)} birimden sözlükte olmayan: {sorted(set(s.index) - kodlar)}")
        print(f"  sözlükteki grup değerleri: {sorted(sz['grup'].dropna().unique())}")

rapor["birim_sayilari"] = {"glim": len(gl), "ay": len(ay), "mars": len(ma)}
(KAYIT / "girdi_denetim.json").write_text(json.dumps(rapor, ensure_ascii=False, indent=2), encoding="utf-8")
print("\n→ kayit\\girdi_denetim.json, kayit\\sude_jeoloji_sablon.csv (verideki bütün birimler, boş açıklama/grup/kaynak sütunlarıyla)")

=== Dosyalar ===
YOK   hedefler.csv
YOK   jeoloji_sozluk.csv
YOK   dogrulama_ciftleri.csv
YOK   v10_lambert_reimold.csv
YOK   analog_metin.csv
YOK   metin.json

=== hedefler.csv içeriği ===
  dosya yok

=== Verideki jeoloji birimleri === GLiM 16 sınıf: ['su', 'ss', 'sm', 'mt', 'ig', 'sc', 'pa', 'vb', 'vi', 'va', 'wb', 'pb', 'py', 'pi', 'ev', 'nd']
Ay 49 birim; belge 6.3 mare listesi verideki: ['Im1', 'Im2', 'Em', 'Imd', 'Id', 'EIp'], verideki değil: []
Mars 44 birim: ['mNh', 'lHl', 'eNh', 'AHv', 'lNh', 'AHi', 'eHv', 'eHt', 'lAv', 'mAl', 'HNt', 'lHv', 'lHt', 'lNv', 'AHtu', 'Av', 'Apu', 'eNhm', 'mNhm', 'eHh', 'Nhu', 'Hto', 'Hp', 'HNhu', 'Aa', 'lHb', 'Ht', 'Ave', 'Htu', 'lApc', 'HNb', 'eAb', 'lHvf', 'eHb', 'Hve', 'ANa', 'lAvf', 'lApd', 'lAa', 'Hpe', 'Ap', 'Nhe', 'Nve', 'Hpu']

→ kayit\girdi_denetim.json, kayit\sude_jeoloji_sablon.csv (verideki bütün birimler, boş açıklama/grup/kaynak sütunlarıyla)


In [13]:
# Developed with assistance from Claude (Anthropic).
# Sude'nin girdi dosyalarının belgeden doğrudan aktarılabilen kısmı (belge 6.3, 9.1–9.3; indirme_raporu3 B5).
# Eksik kalanlar boş bırakılır; Sude doldurur. Var olan dosyanın üzerine yazmaz.
import json, datetime
from pathlib import Path
import pandas as pd, pyogrio

KOK = Path.home() / "orbitrary"; KOD = KOK / "kod"; KAYIT = KOK / "kayit"; TEMIZ = KOK / "temiz"
for ad in ("hedefler.csv", "dogrulama_ciftleri.csv", "jeoloji_sozluk.csv"):
    assert not (KOD / ad).exists(), f"{ad} zaten var, üzerine yazılmaz"

B5 = "approximate centre (mean of candidate landing sites, Wueller et al. 2026)"
B5_YER = "indirme_raporu3.md B5; Wueller ve ark. 2026 JGR Planets 131 e2025JE009434, Zenodo doi:10.5281/zenodo.17084058"
ART = "Artemis III candidate regions (Oct 2024) — first landing now planned as Artemis IV"
H = [  # kimlik, ad, cisim, enlem, boylam, kaynak, kaynak_yeri, not
 ("A3-01", "Peak near Cabeus B", "ay", -83.576, -69.076, B5, B5_YER, ART),
 ("A3-02", "Haworth", "ay", -86.799, -22.898, B5, B5_YER, ART),
 ("A3-03", "Malapert Massif", "ay", -86.047, 0.149, B5, B5_YER, ART),
 ("A3-04", "Mons Mouton Plateau", "ay", -84.534, 30.655, B5, B5_YER, ART),
 ("A3-05", "Mons Mouton", "ay", -85.399, 31.180, B5, B5_YER, ART + "; bölge merkezi IAU zirvesinden ~24 km güneyde (B5)"),
 ("A3-06", "Nobile Rim 1", "ay", -85.188, 37.240, B5, B5_YER, ART),
 ("A3-07", "Nobile Rim 2", "ay", -83.888, 58.403, B5, B5_YER, ART),
 ("A3-08", "de Gerlache Rim 2", "ay", -88.174, -68.787, B5, B5_YER, ART),
 ("A3-09", "Slater Plain", "ay", -87.207, 126.606, B5, B5_YER, ART),
 ("M-SP", "Ay güney kutbu test noktası", "ay", -89.0, 30.0, "Proje test noktası", "belge 9.1", "V1'deki iki çift ona bağlı"),
 ("M-A11", "Apollo 11, Mare Tranquillitatis", "ay", None, None, "NSSDCA / Wagner ve ark. 2017 [141]", "", "Sude doldurur"),
 ("M-A14", "Apollo 14, Fra Mauro", "ay", -3.64589, -17.47194, "NSSDCA / Wagner ve ark. 2017 [141]", "belge 9.1", "V1"),
 ("M-A15", "Apollo 15, Hadley–Apennine", "ay", 26.13239, 3.63330, "NSSDCA / Wagner ve ark. 2017 [141]", "belge 9.1", "V1"),
 ("M-A16", "Apollo 16, Descartes", "ay", -8.9734, 15.5011, "NSSDCA / Wagner ve ark. 2017 [141]", "belge 9.2 tablosu", "V1"),
 ("M-A17", "Apollo 17, Taurus–Littrow", "ay", 20.1911, 30.7723, "NSSDCA / Wagner ve ark. 2017 [141]", "belge 9.2 tablosu", "V1"),
 ("M-SHK", "Shackleton krateri", "ay", None, None, "IAU Gazetteer [140]", "", "Sude doldurur"),
 ("R-JEZ", "Jezero (Perseverance)", "mars", 18.44, 77.45, "NASA/JPL", "belge 9.2", "belge: Sude kaynağı doğrular"),
 ("R-GAL", "Gale (Curiosity)", "mars", -4.5895, 137.4417, "NASA/JPL", "belge 9.2", "V1"),
 ("R-PF", "Ares Vallis (Pathfinder)", "mars", 19.13, -33.22, "Golombek ve ark. 1999 [142]", "belge 9.2", "V1"),
 ("R-GUS", "Gusev (Spirit)", "mars", None, None, "NASA/JPL", "", "Sude doldurur"),
 ("R-MER", "Meridiani Planum (Opportunity)", "mars", None, None, "NASA/JPL", "", "Sude doldurur"),
 ("R-PHX", "Phoenix iniş yeri", "mars", None, None, "NASA/JPL", "", "Sude doldurur; Ice sonucu yok (60° ötesi)"),
 ("R-UTO", "Utopia Planitia (Zhurong / Viking 2)", "mars", None, None, "", "", "Sude doldurur (kaynaklı)"),
 ("R-ELY", "Elysium Planitia (InSight)", "mars", None, None, "NASA/JPL", "", "Sude doldurur"),
 ("R-OLY", "Olympus Mons zirvesi", "mars", None, None, "IAU Gazetteer [140]", "", "Sude doldurur"),
]
hd = pd.DataFrame(H, columns=["kimlik", "ad", "cisim", "enlem", "boylam", "kaynak", "kaynak_yeri", "not"])
hd.insert(5, "boylam_yonu", "dogu+ (-180…180)")
hd.to_csv(KOD / "hedefler.csv", index=False, encoding="utf-8-sig")

RG, SB, KH = (36.47611, -105.73222), (46.600, -81.183), (31.97194, -106.96472)
V1 = [  # cift_no, hedef_kimlik, dunya_saha, (enlem, boylam), gorev, kaynak, ilk_cift
 (1, "M-A15", "Rio Grande Gorge, Taos, New Mexico", RG, "geology", "NASA History; BLM 2021", True),
 (2, "M-A14", "Nördlinger Ries, Almanya", (48.8833, 10.6167), "geology", "NASA ALSJ Apollo 14 Ries geology trip (10–15 Ağustos 1970); Phinney, NASA SP-2015-626 [143]", True),
 (3, "M-A16", "Sudbury, Ontario", SB, "geology", "NASA History; GSA 2008", True),
 (4, "M-A17", "Sudbury, Ontario", SB, "geology", "CBC 2021", False),
 (5, "M-SP", "SP Crater, Arizona", (35.58223, -111.63182), "mobility", "JETT3 NTRS 20230000146 [144]; JETT5 NTRS 20250000177 [145]", True),
 (6, "R-PF", "Ephrata fanı, Washington", (47.318, -119.553), "genel", "Golombek ve ark. 1999 [142]", True),
 (7, "R-GAL", "Dumont Dunes, Mojave", (35.67863, -116.23156), "mobility", "JPL, 11 Mayıs 2012 (Scarecrow testleri) [146]", True),
 (8, "M-A15", "Kilbourne Hole, New Mexico", KH, "geology", "BLM 2021", True),
 (9, "M-A16", "Rio Grande Gorge, Taos, New Mexico", RG, "geology", "NASA History (Taos, Eylül 1971)", False),
 (10, "M-A17", "Lunar Crater Volcanic Field, Nevada", (38.40, -116.05), "geology", "Phinney, NASA SP-2015-626 [143]", True),
 (11, "M-SP", "Kilbourne Hole, New Mexico", KH, "mobility", "JETT1 NTRS 20220015570 [147]", False),
]
dc = pd.DataFrame([{"tur": "V1", "cift_no": c, "hedef_kimlik": h, "dunya_saha": s, "enlem": ll[0], "boylam": ll[1],
                    "gorev": g, "kaynak": k, "ilk_cift": i, "kaynak_yeri": "belge 9.3"} for c, h, s, ll, g, k, i in V1])
assert len(dc) == 11 and dc["ilk_cift"].sum() == 8  # belge 9.3: bağımsız 8 çift
dc.to_csv(KOD / "dogrulama_ciftleri.csv", index=False, encoding="utf-8-sig")

GLIM = {"va": "volkanik", "vi": "volkanik", "vb": "volkanik", "py": "volkanik", "pb": "bazik_plutonik",
        "pa": "asidik_ara_plutonik", "pi": "asidik_ara_plutonik", "su": "tortul", "ss": "tortul", "sm": "tortul",
        "sc": "tortul", "ev": "tortul", "mt": "metamorfik", "ig": "buz", "wb": "hesap_disi", "nd": "hesap_disi"}
MARE = {"Im1", "Im2", "Em", "Imd", "Id", "EIp"}
def birim_listesi(gpkg, katman):
    return sorted(pyogrio.read_dataframe(gpkg, layer=katman, columns=["birim"], read_geometry=False)["birim"].astype(str).unique())
gl = birim_listesi(next((TEMIZ / "dunya").glob("E3_*")) / "glim.gpkg", "glim")
ay = birim_listesi(next((TEMIZ / "ay").glob("M2_*")) / "ay_jeoloji.gpkg", "ay_jeoloji")
ma = birim_listesi(next((TEMIZ / "mars").glob("R2_*")) / "sim3292.gpkg", "sim3292")
assert set(gl) == set(GLIM), set(gl) ^ set(GLIM)
sat = [("dunya", b, "", GLIM[b], "belge 6.3 (GLiM [47] sınıf → grup); Sude açıklamayı ve pb kapsamını GLiM makalesinden doğrular") for b in gl]
sat += [("ay", b, "", "volkanik" if b in MARE else "bazik_plutonik", "belge 6.3 (mare birimleri volkanik, diğerleri bazik plütonik)") for b in ay]
sat += [("mars", b, "", "", "") for b in ma]
sz = pd.DataFrame(sat, columns=["cisim", "birim_kodu", "birim_aciklama", "grup", "kaynak"])
sz.to_csv(KOD / "jeoloji_sozluk.csv", index=False, encoding="utf-8-sig")

with open(KAYIT / "14_kasim_log.txt", "a", encoding="utf-8") as f:
    f.write(f"{datetime.datetime.now().isoformat(timespec='seconds')}\tgirdi dosyaları belgeden (hedefler 17/25, V1 11, sözlük Dünya+Ay; Mars 44 boş)\tOK\n")
print(f"hedefler.csv: {len(hd)} satır, koordinatlı {hd['enlem'].notna().sum()}, boş {hd['enlem'].isna().sum()}: {list(hd.loc[hd['enlem'].isna(), 'kimlik'])}")
print(f"dogrulama_ciftleri.csv: {len(dc)} V1 çifti (ilk çift {int(dc['ilk_cift'].sum())})")
print(f"jeoloji_sozluk.csv: Dünya {len(gl)}, Ay {len(ay)} (mare {len(MARE & set(ay))}), Mars {len(ma)} birim — Mars grubu boş")

hedefler.csv: 25 satır, koordinatlı 17, boş 8: ['M-A11', 'M-SHK', 'R-GUS', 'R-MER', 'R-PHX', 'R-UTO', 'R-ELY', 'R-OLY']
dogrulama_ciftleri.csv: 11 V1 çifti (ilk çift 8)
jeoloji_sozluk.csv: Dünya 16, Ay 49 (mare 6), Mars 44 birim — Mars grubu boş


In [ ]:
%run C:/Users/USER/orbitrary/kod/hucre_ozellik_diger.py

In [1]:
# Developed with assistance from Claude (Anthropic).
# Sude'nin dönüşünü birleştir (14 Kasım): hedefler 8 + R-JEZ kaynağı, Mars sözlüğü 44, V1 kaynak notları; kararlar uygulanır.
import json, shutil, sys, subprocess, datetime
from pathlib import Path
import pandas as pd

KOK = Path.home() / "orbitrary"; KOD = KOK / "kod"; KAYIT = KOK / "kayit"
zaman = datetime.datetime.now().strftime("%Y%m%d_%H%M")
for ad in ("hedefler.csv", "jeoloji_sozluk.csv", "dogrulama_ciftleri.csv"):
    shutil.copy2(KOD / ad, KAYIT / f"yedek_{zaman}_{ad}")
oku = lambda p: pd.read_csv(p, dtype=str, encoding="utf-8-sig").fillna("")

# --- 1. hedefler.csv
hd = oku(KOD / "hedefler.csv").set_index("kimlik")
sd = oku(KOD / "hedefler_doldurma.csv").set_index("kimlik")
assert set(sd.index) <= set(hd.index), set(sd.index) - set(hd.index)
for k in sd.index:
    if k == "R-JEZ":   # karar 1: belge değeri kalır; Wikipedia değeri kullanılmaz
        hd.loc[k, "kaynak_yeri"] = "belge 9.2; JPL Mars 2020 basın kiti ('yaklaşık 18° enlem, 77° boylam'): https://www.jpl.nasa.gov/news/press_kits/mars_2020/launch/mission/landing_site/"
        hd.loc[k, "not"] = "Belge değeri 18,44 / 77,45; NASA/JPL'de hassas değer yok, Wikipedia değeri kullanılmadı (14 Kasım kararı)"
        continue
    for c in ("enlem", "boylam", "kaynak", "kaynak_yeri", "not"):
        hd.loc[k, c] = sd.loc[k, c]
# karar 2: Utopia iki hedef
hd.loc["R-UTO", ["ad", "enlem", "boylam", "kaynak", "kaynak_yeri", "not"]] = [
    "Utopia Planitia (Viking 2)", "47.64", "134.29", "NSSDCA Viking (planetosentrik)",
    "https://nssdc.gsfc.nasa.gov/planetary/viking.html", "Belgedeki 'Zhurong / Viking 2' iki hedefe ayrıldı (14 Kasım kararı); değer Sude'nin raporundan"]
zhu = pd.DataFrame([{"ad": "Utopia Planitia (Zhurong)", "cisim": "mars", "enlem": "25.066", "boylam": "109.925",
                     "boylam_yonu": hd.loc["R-UTO", "boylam_yonu"],
                     "kaynak": "Liu ve ark. 2022, Nature Astronomy, doi:10.1038/s41550-021-01519-5 (HiRIC görüntüsünden)",
                     "kaynak_yeri": "https://link.springer.com/10.1038/s41550-021-01519-5 (Özet)",
                     "not": "Belgedeki 'Zhurong / Viking 2' iki hedefe ayrıldı (14 Kasım kararı)"}], index=pd.Index(["R-ZHU"], name="kimlik"))
hd = pd.concat([hd, zhu]) if "R-ZHU" not in hd.index else hd
# karar 3: Olympus Mons özellik merkezi
hd.loc["R-OLY", "ad"] = "Olympus Mons (feature centre, IAU Gazetteer)"
hd.loc["R-OLY", "not"] = hd.loc["R-OLY", "not"] + " | sitede 'feature centre (IAU Gazetteer)'; 'zirve' denmez (14 Kasım kararı)"
hd.reset_index().to_csv(KOD / "hedefler.csv", index=False, encoding="utf-8-sig")
bos = hd.index[(hd["enlem"] == "") | (hd["boylam"] == "")].tolist()
assert not bos, f"koordinatı boş hedef kaldı: {bos}"
for c in ("enlem", "boylam"):
    v = hd[c].astype(float); assert v.abs().max() <= (90 if c == "enlem" else 180), c

# --- 2. jeoloji_sozluk.csv (Mars 44; GLiM pb gerekçesi)
sz = oku(KOD / "jeoloji_sozluk.csv")
sm = oku(KOD / "jeoloji_sozluk_mars.csv")
sm["birim_kodu"] = sm["birim_kodu"].str.strip()
mars = sz["cisim"].str.lower() == "mars"
assert set(sm["birim_kodu"]) == set(sz.loc[mars, "birim_kodu"]) and len(sm) == 44 and not sm["birim_kodu"].duplicated().any()
sm = sm.set_index("birim_kodu")
for i in sz.index[mars]:
    b = sz.loc[i, "birim_kodu"]
    for c in ("birim_aciklama", "grup", "kaynak"):
        sz.loc[i, c] = sm.loc[b, c]
pb = (sz["cisim"] == "dunya") & (sz["birim_kodu"] == "pb")
sz.loc[pb, "kaynak"] = ("belge 6.3; Hartmann & Moosdorf 2012 (G3 13, Q12004) Ek A4.1.4, paragraf [71]: pb = gabro, norit, "
                        "peridotit; anortozit metinde geçmez, en yakın sınıf olarak pb kullanıldı (Sude, 14 Kasım)")
sz.to_csv(KOD / "jeoloji_sozluk.csv", index=False, encoding="utf-8-sig")
print("Mars grupları:", sz.loc[mars, "grup"].value_counts().to_dict())

# --- 3. dogrulama_ciftleri.csv: kaynak notları (çiftler değişmez)
dc = oku(KOD / "dogrulama_ciftleri.csv")
dc.loc[dc["cift_no"] == "5", "kaynak"] += ("; JETT5 USGS: https://www.usgs.gov/centers/astrogeology-science-center/news/"
                                           "artemis-iii-mission-tests-san-francisco-volcanic-field (NTRS 20250000177 kimliği doğrulanamadı, Sude)")
dc.loc[dc["cift_no"] == "7", "kaynak"] += "; testler Mayıs 2012 başı, 11 Mayıs JPL haberinin tarihi (Sude)"
dc.loc[dc["cift_no"] == "11", "kaynak"] += "; kaynakta kesin tarih yok, JETT serisi Nisan 2022'de başladı (Sude)"
dc["sude_kontrol"] = "kaynakla tutuyor (14 Kasım)"
dc.to_csv(KOD / "dogrulama_ciftleri.csv", index=False, encoding="utf-8-sig")

# --- kayıt
deg = KAYIT / "degisiklik.md"
satirlar = [
    "- 2026-11-14 | Sude dönüşü birleştirildi: 8 hedef koordinatı, Mars sözlüğü 44 birim (12 volkanik, 16 tortul, 7 buz, 9 yayla_yarim), V1 11 çift kaynakla tutuyor | yedekler kayit\\yedek_*\\n",
    "- 2026-11-14 | R-JEZ belge değeri (18,44 / 77,45) kaldı; Wikipedia hassas değeri kullanılmadı | etkisi: yok\\n",
    "- 2026-11-14 | R-UTO ikiye ayrıldı: R-UTO Viking 2 (47,64 / 134,29, NSSDCA), R-ZHU Zhurong (25,066 / 109,925, Liu ve ark. 2022) | neden: iki iniş yeri ~1600 km uzak | etkisi: hedef tablosu (belge 9.2)\\n",
    "- 2026-11-14 | R-OLY = IAU Gazetteer özellik merkezi, 'feature centre' etiketi | etkisi: hedef tablosu, site\\n",
    "- 2026-11-14 | Açık kontrol 'GLiM pb kapsamı' kapandı (kısmen): pb = gabro/norit/peridotit, anortozit geçmez; Ay → pb eşlemesi değişmez, gerekçe cümlesi düzeltildi | etkisi: Method 6.3, Limitations\\n",
    "- 2026-11-14 | lApd, Hpe, Nhe belge 6.3 kuralıyla gruplandı (DMU tanımından farklı); kural değiştirilmedi | etkisi: Limitations\\n",
]
d = deg.read_text(encoding="utf-8")
deg.write_text(d + "".join(s for s in satirlar if s.split(" | ")[1][:30] not in d), encoding="utf-8")
(KAYIT / "sude_birlestirme.json").write_text(json.dumps({
    "zaman": zaman, "hedef_sayisi": len(hd), "hedefler": hd[["ad", "cisim", "enlem", "boylam"]].reset_index().to_dict("records"),
    "mars_gruplari": sz.loc[mars, "grup"].value_counts().to_dict()}, ensure_ascii=False, indent=1), encoding="utf-8")

r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "testler/test_ozellik_diger_gercek.py", "-k", "mars_jeoloji"],
                   cwd=KOD, capture_output=True, text=True)
print(r.stdout[-1500:] + r.stderr[-500:])
with open(KAYIT / "14_kasim_log.txt", "a", encoding="utf-8") as f:
    f.write(f"{datetime.datetime.now().isoformat(timespec='seconds')}\tSude birleştirme + mars_jeoloji testi\t{'PASS' if r.returncode == 0 else 'FAIL'}\n")
print(f"hedefler.csv: {len(hd)} hedef, koordinatı boş: {bos or 'yok'}")

Mars grupları: {'tortul': 16, 'volkanik': 12, 'yayla_yarim': 9, 'buz': 7}
.                                                                        [100%]
1 passed, 42 deselected in 2.15s

hedefler.csv: 26 hedef, koordinatı boş: yok


In [ ]:
   %run C:/Users/USER/orbitrary/kod/hucre_bellek_testkos.py

============================= test session starts =============================
platform win32 -- Python 3.12.16, pytest-9.2.0, pluggy-1.7.0 -- C:\Users\USER\anaconda3\envs\orbitrary\python.exe
rootdir: C:\Users\USER\orbitrary\kod
plugins: anyio-4.16.0, platformdirs-4.14.5
collecting ... collected 441 items

testler/test_ayar.py::test_klasorler_var PASSED                          [  0%]
testler/test_ayar.py::test_her_parametrede_kaynak PASSED                 [  0%]
testler/test_ayar.py::test_belirlenmemis_deger_okunamaz PASSED           [  0%]
testler/test_ayar.py::test_gorev_agirligi_8_2 PASSED                     [  0%]
testler/test_ayar.py::test_tablo_7_3_formulle_ayni PASSED                [  1%]
testler/test_ayar.py::test_esit_sema PASSED                              [  1%]
testler/test_ayar.py::test_tohum_ve_tekrar PASSED                        [  1%]
testler/test_ayar.py::test_sabitler_4_1 PASSED                           [  1%]
testler/test_ayar.py::test_analog_uzaklik_v121 PAS

In [ ]:
# Developed with assistance from Claude (Anthropic).
# Teşhis: (a) PGDA dosya adı düzeltmesini katman.py'ye yeniden işle, (b) CMG değer aralıkları, (c) Apollo 15 Ay penceresi kategori kategori süre.
import sys, json, time, math, importlib, datetime
from pathlib import Path
KOK = Path.home() / "orbitrary"; KOD, KAYIT, ARA = KOK / "kod", KOK / "kayit", KOK / "ara"

# (a) PGDA dosya adı (10:04 düzeltmesi, bellek hücresinde kaybolmuştu)
yol = KOD / "katman.py"; s = yol.read_text(encoding="utf-8")
eski = '''        kutup = ad[-1]
        ds = rasterio.open(_kl("ay", "M7") / f"{ad}.tif")'''
yeni = '''        kutup = ad[-1]
        dosya = _kl("ay", "M7") / f"{ad[:-2]}_65{kutup}.tif"   # avgvisib_65n.tif, lpsr_65s.tif …
        assert dosya.is_file(), dosya
        ds = rasterio.open(dosya)'''
if "_65{kutup}" in s:
    print("(a) PGDA adı zaten düzgün")
else:
    assert s.count(eski) == 1; yol.write_text(s.replace(eski, yeni), encoding="utf-8"); print("(a) PGDA adı yeniden düzeltildi")
with open(KAYIT / "degisiklik.md", "a", encoding="utf-8") as f:
    f.write("- 2026-11-14 | PGDA dosya adı düzeltmesi yeniden işlendi (bellek hücresi katman.py'yi eski kopyadan yazmıştı; 11:41 koşusunda 9 FAIL) | etkisi: yok (kod)\n")

# (b) CMG aralıkları (test_dunya_cmg_turetilmis için)
print("\n(b) CMG ızgaraları:")
for ad in ("mi", "tmax", "tort", "tgece", "iarti"):
    j = json.loads((ARA / f"cmg_{ad}.json").read_text(encoding="utf-8"))
    print(f"   {ad:6s} geçerli {j['gecerli']:>10,d} | en küçük {j['en_kucuk']:.4g} | en büyük {j['en_buyuk']:.4g}")

# (c) Apollo 15 Ay penceresi, kategori kategori
if str(KOD) not in sys.path: sys.path.insert(0, str(KOD))
import katman, ozellik_diger as O
importlib.reload(katman); importlib.reload(O)
print("\n(c) Ay M-A15 (26,13239; 3,63330):", flush=True)
for k in (3, 4, 5, 6, 7, 8):
    t = time.time(); print(f"   kategori {k} başladı {datetime.datetime.now():%H:%M:%S}", flush=True)
    T = O.pencere_temsili("ay", [math.radians(26.13239)], [math.radians(3.63330)], kategoriler=(k,), n_ref=20000)
    print(f"   kategori {k} bitti: {time.time() - t:.1f} s | anahtarlar {sorted(T)}", flush=True)

(a) PGDA adı yeniden düzeltildi

(b) CMG ızgaraları:
   mi     geçerli  8,618,147 | en küçük 0.3971 | en büyük 4.249
   tmax   geçerli  8,861,664 | en küçük 229.8 | en büyük 342.3
   tort   geçerli  8,860,628 | en küçük 209.9 | en büyük 316.7
   tgece  geçerli  8,861,365 | en küçük 209 | en büyük 305.7
   iarti  geçerli  8,638,888 | en küçük 0.0003067 | en büyük 75.28

(c) Ay M-A15 (26,13239; 3,63330):
   kategori 3 başladı 11:48:28
   kategori 3 bitti: 1.4 s | anahtarlar ['q3', 'q3_gecerli']
   kategori 4 başladı 11:48:30
   kategori 4 bitti: 0.1 s | anahtarlar ['Q4', 'x4_gecerli']
   kategori 5 başladı 11:48:30


In [1]:
   %run C:/Users/USER/orbitrary/kod/hucre_duzeltme_1155.py

I⊕ teşhis: {'iarti_gecerli_piksel': 8638888, 'dTg_1K_alti_ayi_olan_piksel': 2913886, 'oran': 0.3289, 'iarti_yuzdelikler': {1: 0.0112, 50: 0.0541, 99: 0.5152, 99.9: 3.9962}}
============================= test session starts =============================
platform win32 -- Python 3.12.16, pytest-9.2.0, pluggy-1.7.0 -- C:\Users\USER\anaconda3\envs\orbitrary\python.exe
rootdir: C:\Users\USER\orbitrary\kod
plugins: anyio-4.16.0, platformdirs-4.14.5
collecting ... collected 442 items

testler/test_ayar.py::test_klasorler_var PASSED                          [  0%]
testler/test_ayar.py::test_her_parametrede_kaynak PASSED                 [  0%]
testler/test_ayar.py::test_belirlenmemis_deger_okunamaz PASSED           [  0%]
testler/test_ayar.py::test_gorev_agirligi_8_2 PASSED                     [  0%]
testler/test_ayar.py::test_tablo_7_3_formulle_ayni PASSED                [  1%]
testler/test_ayar.py::test_esit_sema PASSED                              [  1%]
testler/test_ayar.py::test_tohum_ve_t

In [2]:
%run C:/Users/USER/orbitrary/kod/hucre_sude_iarti.py

1) D2.csv'de 0,604 zaten yazılı:
govde set                    test nokta enlem boylam koordinat_kaynagi  beklenen                                                                       tolerans_kurali                                                                                                                            beklenen_kaynagi                                                olculen durum               zaman
dunya  E4 gecerli yayinim araligi                                      0,61-1,00 min >= 0,61 PASS; 0,52-0,61 UYARI (kayda yazilir, is surer); <= 0,51 FAIL (talimat 7) v1.0 indirmesinde ölçülen en küçük değer 0,612; Hulley ve Hook 2009 (HyspIRI Workshop) kum tepesi sahalarındaki 0,6–0,96 aralığıyla tutarlı en kucuk 0.604 (DN 57), en buyuk 1.000 (DN 255), 12 ay UYARI 2026-11-14T00:13:33
   ay 01: ΔTg ≤ 0 ve C₁ > 0 piksel 108,029
   ay 02: ΔTg ≤ 0 ve C₁ > 0 piksel 226,275
   ay 03: ΔTg ≤ 0 ve C₁ > 0 piksel 134,462
   ay 04: ΔTg ≤ 0 ve C₁ > 0 piksel 36,406
   ay 05: ΔTg ≤ 0 ve

In [3]:
%run C:/Users/USER/orbitrary/kod/hucre_kapi.py

============================= test session starts =============================
platform win32 -- Python 3.12.16, pytest-9.2.0, pluggy-1.7.0 -- C:\Users\USER\anaconda3\envs\orbitrary\python.exe
rootdir: C:\Users\USER\orbitrary\kod
plugins: anyio-4.16.0, platformdirs-4.14.5
collecting ... collected 16 items

testler/test_kapi.py::test_kara_maskesi_8_3 PASSED                       [  6%]
testler/test_kapi.py::test_hareket_kapisi_8_2 PASSED                     [ 12%]
testler/test_kapi.py::test_buz_kapisi_8_2 PASSED                         [ 18%]
testler/test_kapi.py::test_kapi_yedegi_8_2 PASSED                        [ 25%]
testler/test_kapi.py::test_astrobiyoloji_kapisi_8_2 PASSED               [ 31%]
testler/test_kapi.py::test_bitki_ortusu_8_2 PASSED                       [ 37%]
testler/test_kapi.py::test_eksik_kapi_verisi_8_2 PASSED                  [ 43%]
testler/test_kapi.py::test_kuru_hedefte_ice_uygulanmaz PASSED            [ 50%]
testler/test_kapi.py::test_w_kare_kara_esigi_K1 PAS

In [4]:
   %run C:/Users/USER/orbitrary/kod/hucre_teorem.py

============================= test session starts =============================
platform win32 -- Python 3.12.16, pytest-9.2.0, pluggy-1.7.0 -- C:\Users\USER\anaconda3\envs\orbitrary\python.exe
rootdir: C:\Users\USER\orbitrary\kod
plugins: anyio-4.16.0, platformdirs-4.14.5
collecting ... collected 34 items

testler/test_teorem.py::test_ornek_buyukluk_8_1 PASSED                   [  2%]
testler/test_teorem.py::test_lemma1_hucre_sinirlari[0] PASSED            [  5%]
testler/test_teorem.py::test_lemma1_hucre_sinirlari[1] PASSED            [  8%]
testler/test_teorem.py::test_lemma1_hucre_sinirlari[2] PASSED            [ 11%]
testler/test_teorem.py::test_lemma1_hucre_sinirlari[3] PASSED            [ 14%]
testler/test_teorem.py::test_lemma1_hucre_sinirlari[4] PASSED            [ 17%]
testler/test_teorem.py::test_lemma1_hucre_sinirlari[5] PASSED            [ 20%]
testler/test_teorem.py::test_lemma1_hucre_sinirlari[6] PASSED            [ 23%]
testler/test_teorem.py::test_lemma1_hucre_sinirlari

In [5]:
   %run C:/Users/USER/orbitrary/kod/hucre_onele.py

ayar.json Kön/recall yaprakları: [('onele.K_on_adaylari', [100, 200, 400, 800]), ('onele.K_on', None), ('onele.recall_esigi', 0.9)]
============================= test session starts =============================
platform win32 -- Python 3.12.16, pytest-9.2.0, pluggy-1.7.0 -- C:\Users\USER\anaconda3\envs\orbitrary\python.exe
rootdir: C:\Users\USER\orbitrary\kod
plugins: anyio-4.16.0, platformdirs-4.14.5
collecting ... collected 15 items

testler/test_onele.py::test_z_vektoru_bilesenleri PASSED                 [  6%]
testler/test_onele.py::test_z_vektoru_dem_eksik_nan PASSED               [ 13%]
testler/test_onele.py::test_medyan_mad_elle PASSED                       [ 20%]
testler/test_onele.py::test_mad_sifir_bilesen_cikar_Q1 PASSED            [ 26%]
testler/test_onele.py::test_kd_kaba_kuvvetle_ayni[1] PASSED              [ 33%]
testler/test_onele.py::test_kd_kaba_kuvvetle_ayni[5] PASSED              [ 40%]
testler/test_onele.py::test_kd_kaba_kuvvetle_ayni[37] PASSED             [ 46%]

In [6]:
   %run C:/Users/USER/orbitrary/kod/hucre_onele_q9.py

============================= test session starts =============================
platform win32 -- Python 3.12.16, pytest-9.2.0, pluggy-1.7.0 -- C:\Users\USER\anaconda3\envs\orbitrary\python.exe
rootdir: C:\Users\USER\orbitrary\kod
plugins: anyio-4.16.0, platformdirs-4.14.5
collecting ... collected 16 items

testler/test_onele.py::test_z_vektoru_bilesenleri PASSED                 [  6%]
testler/test_onele.py::test_z_vektoru_dem_eksik_nan PASSED               [ 12%]
testler/test_onele.py::test_medyan_mad_elle PASSED                       [ 18%]
testler/test_onele.py::test_mad_sifir_bilesen_cikar_Q1 PASSED            [ 25%]
testler/test_onele.py::test_kd_kaba_kuvvetle_ayni[1] PASSED              [ 31%]
testler/test_onele.py::test_kd_kaba_kuvvetle_ayni[5] PASSED              [ 37%]
testler/test_onele.py::test_kd_kaba_kuvvetle_ayni[37] PASSED             [ 43%]
testler/test_onele.py::test_kd_kaba_kuvvetle_ayni[200] PASSED            [ 50%]
testler/test_onele.py::test_esitlikte_kucuk_kimlik_

In [7]:
   %run C:/Users/USER/orbitrary/kod/hucre_iarti_29.py

silindi: cmg_iarti.npy
silindi: cmg_iarti.json
cmg_iarti yeniden üretiliyor (12 ay LST + albedo)...
bitti: 9 s
cmg_iarti.json: geçerli 8638888 en küçük 0.000306722562527284 en büyük 75.2815170288086
   ay 01
   ay 02
   ay 03
   ay 04
   ay 05
   ay 06
   ay 07
   ay 08
   ay 09
   ay 10
   ay 11
   ay 12
{
 "gecerli_piksel": 8638888,
 "yeni_en_kucuk": 0.000306722562527284,
 "yeni_en_buyuk": 75.2815170288086,
 "sira_farki": {
  "ortanca": 0.0206,
  "yuzde95": 0.2282,
  "en_buyuk": 0.9883
 }
}
============================= test session starts =============================
platform win32 -- Python 3.12.16, pytest-9.2.0, pluggy-1.7.0 -- C:\Users\USER\anaconda3\envs\orbitrary\python.exe
rootdir: C:\Users\USER\orbitrary\kod
plugins: anyio-4.16.0, platformdirs-4.14.5
collecting ... collected 25 items

testler/test_ozellik_diger.py::test_C1_8_2[0-0-1.5707963267948966] PASSED [  4%]
testler/test_ozellik_diger.py::test_C1_8_2[45-20-1.5165] PASSED          [  8%]
testler/test_ozellik_diger.py::t

In [8]:
   %run C:/Users/USER/orbitrary/kod/hucre_puan.py

ayar.json puan yaprakları:
   cisim.yaricap_km = {'dunya': 6371.0, 'ay': 1737.4, 'mars': 3389.5}
   izgara.Delta_km = 20.0
   izgara.L_km = 20.0
   referans.katman_A = {'1': 2000, '2': 200, '3': "E'", '4': "E'", '5': "E'", '6': "E'", '7': "E'", '8': 200}
   referans.katman_B = {'1': 2000, '2': 200, '3': 2000, '4': 2000, '5': 2000, '6': 2000, '7': 2000, '8': 200}
   referans.en_kucuk = 100
   topografya.olcekler = [1, 4]
   topografya.IQR_yuzdelikleri = [0.25, 0.75]
   topoloji.kappa = 2
   topoloji.aday_sayisi = 30
   onele.K_on_adaylari = [100, 200, 400, 800]
   agirlik.ana_pay = 0.8
   agirlik.ana_kategoriler = {'mobility': [1, 2, 7], 'geology': [2, 3, 4], 'ice': [5, 6, 8], 'astro': [4, 5, 6]}
   agirlik.duyarlilik_sema = ['esit', 'geometrik']
   siralama.top3_en_az_km = 100.0
   analog_uzaklik_km = {'bilinen_alti': 20.0, 'yakin_ust': 100.0}
============================= test session starts =============================
platform win32 -- Python 3.12.16, pytest-9.2.0, pluggy-1.7.0 -- 

In [9]:
   %run C:/Users/USER/orbitrary/kod/hucre_puan_q11.py

============================= test session starts =============================
platform win32 -- Python 3.12.16, pytest-9.2.0, pluggy-1.7.0 -- C:\Users\USER\anaconda3\envs\orbitrary\python.exe
rootdir: C:\Users\USER\orbitrary\kod
plugins: anyio-4.16.0, platformdirs-4.14.5
collecting ... collected 19 items

testler/test_puan.py::test_gorev_agirligi_8_2 PASSED                     [  5%]
testler/test_puan.py::test_agirlik_tablosu_7_3 PASSED                    [ 10%]
testler/test_puan.py::test_null_dusmesi_8_3 PASSED                       [ 15%]
testler/test_puan.py::test_yuzdelik_kalibrasyon_s1_8_2 PASSED            [ 21%]
testler/test_puan.py::test_yuzdelik_sira_bicimi_8_2 PASSED               [ 26%]
testler/test_puan.py::test_sira_bicimi_formulu_rastgele PASSED           [ 31%]
testler/test_puan.py::test_referans_alt_siniri_8_3 PASSED                [ 36%]
testler/test_puan.py::test_zeta_ve_kendi_haric_ortanca PASSED            [ 42%]
testler/test_puan.py::test_zeta_kaba_kuvvet PASSED 

In [10]:
   %run C:/Users/USER/orbitrary/kod/hucre_dogrula.py

ayar.json alt_tohum: {'R_orneklem': 1, 'V2': 2, 'V3': 3, 'V4_ikinci_R': 4, 'V5': 5, 'V7': 7, 'P_fB': 10, 'katman_B_R': 11}
============================= test session starts =============================
platform win32 -- Python 3.12.16, pytest-9.2.0, pluggy-1.7.0 -- C:\Users\USER\anaconda3\envs\orbitrary\python.exe
rootdir: C:\Users\USER\orbitrary\kod
plugins: anyio-4.16.0, platformdirs-4.14.5
collecting ... collected 14 items

testler/test_dogrula.py::test_pi_ve_K14 PASSED                           [  7%]
testler/test_dogrula.py::test_fisher_elle PASSED                         [ 14%]
testler/test_dogrula.py::test_fisher_H0_altinda_duzgun PASSED            [ 21%]
testler/test_dogrula.py::test_fisher_yon PASSED                          [ 28%]
testler/test_dogrula.py::test_auc_kaba_kuvvet_ve_mw PASSED               [ 35%]
testler/test_dogrula.py::test_v3_tekrar_ve_GA PASSED                     [ 42%]
testler/test_dogrula.py::test_b_prime_ve_hedefler PASSED                 [ 50%]
testler/

In [11]:
   %run C:/Users/USER/orbitrary/kod/hucre_dogrula_q14.py

ayar.json alt_tohum: {'R_orneklem': 1, 'V2': 2, 'V3': 3, 'V4_ikinci_R': 4, 'V5': 5, 'V7': 7, 'P_fB': 10, 'katman_B_R': 11, 'V4_hedef': 6}
============================= test session starts =============================
platform win32 -- Python 3.12.16, pytest-9.2.0, pluggy-1.7.0 -- C:\Users\USER\anaconda3\envs\orbitrary\python.exe
rootdir: C:\Users\USER\orbitrary\kod
plugins: anyio-4.16.0, platformdirs-4.14.5
collecting ... collected 16 items

testler/test_dogrula.py::test_pi_ve_K14 PASSED                           [  6%]
testler/test_dogrula.py::test_fisher_elle PASSED                         [ 12%]
testler/test_dogrula.py::test_fisher_H0_altinda_duzgun PASSED            [ 18%]
testler/test_dogrula.py::test_fisher_yon PASSED                          [ 25%]
testler/test_dogrula.py::test_auc_kaba_kuvvet_ve_mw PASSED               [ 31%]
testler/test_dogrula.py::test_v3_tekrar_ve_GA PASSED                     [ 37%]
testler/test_dogrula.py::test_b_prime_ve_hedefler PASSED                 

In [12]:
   %run C:/Users/USER/orbitrary/kod/hucre_kos1.py

alt_tohum: {'R_orneklem': 1, 'V2': 2, 'V3': 3, 'V4_ikinci_R': 4, 'V5': 5, 'V7': 7, 'P_fB': 10, 'katman_B_R': 11, 'V4_hedef': 6, 'D5_kuru': 12}
kayit\P0.json anahtarları: ['adim', 'zaman', 'makine', 'sure_s', 'd1_satir', 'd1_sha_sayisi', 'temiz_dosya', 'temiz_gb', 'setler', 'sağlama_durum', 'D4']
============================= test session starts =============================
platform win32 -- Python 3.12.16, pytest-9.2.0, pluggy-1.7.0 -- C:\Users\USER\anaconda3\envs\orbitrary\python.exe
rootdir: C:\Users\USER\orbitrary\kod
plugins: anyio-4.16.0, platformdirs-4.14.5
collecting ... collected 4 items

testler/test_kos_hatti.py::test_depo_tam_kosu PASSED                     [ 25%]
testler/test_kos_hatti.py::test_depo_kesilip_devam_K16 PASSED            [ 50%]
testler/test_kos_hatti.py::test_depo_sema_ve_boyut PASSED                [ 75%]
testler/test_kos_hatti.py::test_izgara_sayilari PASSED                   [100%]

============================ slowest 10 durations ========================

In [13]:
   %run C:/Users/USER/orbitrary/kod/hucre_kos_paralel.py

işlemci çekirdeği (mantıksal): 12
RAM (GB): 34.1
============================= test session starts =============================
platform win32 -- Python 3.12.16, pytest-9.2.0, pluggy-1.7.0 -- C:\Users\USER\anaconda3\envs\orbitrary\python.exe
rootdir: C:\Users\USER\orbitrary\kod
plugins: anyio-4.16.0, platformdirs-4.14.5
collecting ... collected 5 items

testler/test_kos_hatti.py::test_depo_tam_kosu PASSED                     [ 20%]
testler/test_kos_hatti.py::test_depo_kesilip_devam_K16 PASSED            [ 40%]
testler/test_kos_hatti.py::test_depo_sema_ve_boyut PASSED                [ 60%]
testler/test_kos_hatti.py::test_izgara_sayilari PASSED                   [ 80%]
testler/test_kos_hatti.py::test_depo_iki_isci FAILED                     [100%]

================================== FAILURES ===================================
_____________________________ test_depo_iki_isci ______________________________

tmp_path = WindowsPath('C:/Users/USER/AppData/Local/Temp/pytest-of-USER/pytest-7/

In [14]:
   %run C:/Users/USER/orbitrary/kod/hucre_kos_utf8.py

============================= test session starts =============================
platform win32 -- Python 3.12.16, pytest-9.2.0, pluggy-1.7.0 -- C:\Users\USER\anaconda3\envs\orbitrary\python.exe
rootdir: C:\Users\USER\orbitrary\kod
plugins: anyio-4.16.0, platformdirs-4.14.5
collecting ... collected 5 items

testler/test_kos_hatti.py::test_depo_tam_kosu PASSED                     [ 20%]
testler/test_kos_hatti.py::test_depo_kesilip_devam_K16 PASSED            [ 40%]
testler/test_kos_hatti.py::test_depo_sema_ve_boyut PASSED                [ 60%]
testler/test_kos_hatti.py::test_izgara_sayilari PASSED                   [ 80%]
testler/test_kos_hatti.py::test_depo_iki_isci PASSED                     [100%]

============================ slowest 10 durations =============================
0.90s call     testler/test_kos_hatti.py::test_depo_iki_isci
0.03s call     testler/test_kos_hatti.py::test_izgara_sayilari
0.02s call     testler/test_kos_hatti.py::test_depo_kesilip_devam_K16
0.02s call     te

In [1]:
%run C:/Users/USER/orbitrary/kod/hucre_p3_kontrol.py

pencere: 346064
  Omega_1        []           {'bicim': [], 'dolu_orani': 1.0}
  Omega_4        []           {'bicim': [], 'dolu_orani': 1.0}
  Q1             [6, 1000]    {'bicim': [6, 1000], 'dolu_orani': 1.0}
  Q4             [1000]       {'bicim': [1000], 'dolu_orani': 0.9996}
  Q5_iarti       [1000]       {'bicim': [1000], 'dolu_orani': 1.0}
  Q5_tgece       [1000]       {'bicim': [1000], 'dolu_orani': 1.0}
  Q5_tmax        [1000]       {'bicim': [1000], 'dolu_orani': 1.0}
  Q5_tort        [1000]       {'bicim': [1000], 'dolu_orani': 1.0}
  Q6             [1000]       {'bicim': [1000], 'dolu_orani': 0.9022}
  Q7             [1000]       {'bicim': [1000], 'dolu_orani': 0.903}
  Q8             [1000]       {'bicim': [1000], 'dolu_orani': 1.0}
  R              []           {'bicim': [], 'dolu_orani': 1.0}
  dem_tam        []           {'bicim': [], 'ortalama': 1.0}
  lnR            []           {'bicim': [], 'dolu_orani': 1.0}
  mob            []           {'bicim': [], 'dolu_orani':

In [2]:
%run C:/Users/USER/orbitrary/kod/hucre_kos2.py

============================= test session starts =============================
platform win32 -- Python 3.12.16, pytest-9.2.0, pluggy-1.7.0 -- C:\Users\USER\anaconda3\envs\orbitrary\python.exe
rootdir: C:\Users\USER\orbitrary\kod
plugins: anyio-4.16.0, platformdirs-4.14.5
collecting ... collected 6 items

testler/test_kos_hatti.py::test_depo_tam_kosu PASSED                     [ 16%]
testler/test_kos_hatti.py::test_depo_kesilip_devam_K16 PASSED            [ 33%]
testler/test_kos_hatti.py::test_depo_sema_ve_boyut PASSED                [ 50%]
testler/test_kos_hatti.py::test_izgara_sayilari PASSED                   [ 66%]
testler/test_kos_hatti.py::test_depo_iki_isci PASSED                     [ 83%]
testler/test_kos_puanla.py::test_uzaklik_ve_puanla PASSED                [100%]

============================ slowest 10 durations =============================
3.78s setup    testler/test_kos_puanla.py::test_uzaklik_ve_puanla
0.94s call     testler/test_kos_puanla.py::test_uzaklik_ve_puanla

In [3]:
%run C:/Users/USER/orbitrary/kod/hucre_kos_main.py

usage: kos.py [-h] [--govde GOVDE] [--n N] [--isci ISCI] [--asama ISCI_ASAMA]
kos.py komut satırı: PASS


In [4]:
%run C:/Users/USER/orbitrary/kod/hucre_kos_diyagram.py

============================= test session starts =============================
platform win32 -- Python 3.12.16, pytest-9.2.0, pluggy-1.7.0 -- C:\Users\USER\anaconda3\envs\orbitrary\python.exe
rootdir: C:\Users\USER\orbitrary\kod
plugins: anyio-4.16.0, platformdirs-4.14.5
collecting ... collected 7 items

testler/test_kos_hatti.py::test_depo_tam_kosu PASSED                     [ 14%]
testler/test_kos_hatti.py::test_depo_kesilip_devam_K16 PASSED            [ 28%]
testler/test_kos_hatti.py::test_depo_sema_ve_boyut PASSED                [ 42%]
testler/test_kos_hatti.py::test_izgara_sayilari PASSED                   [ 57%]
testler/test_kos_hatti.py::test_depo_iki_isci PASSED                     [ 71%]
testler/test_kos_hatti.py::test_diyagram_deposu PASSED                   [ 85%]
testler/test_kos_puanla.py::test_uzaklik_ve_puanla PASSED                [100%]

============================ slowest 10 durations =============================
3.87s setup    testler/test_kos_puanla.py::test_uza

In [5]:
# Developed with assistance from Claude (Anthropic).
# Site sohbetinin yazdığı dosyaları bulur. Hiçbir şey silmez/değiştirmez.
from pathlib import Path
import re, datetime as dt

KOK = Path(r"C:\Users\USER\orbitrary")
SC = KOK / "site_calisma_eski"
assert SC.exists(), "Önce site_calisma klasörünün adını site_calisma_eski yap"
BAS = dt.datetime(2026, 11, 14, 15, 30).timestamp()   # site sohbeti ~15:37'de başladı

print("== 1) Site dosyalarında geçen yollar ==")
for ad in ["site_adim2.py", "site_adim3.py", "kayit_site_adim3.json", "girdi_raporu.json"]:
    f = SC / ad
    if not f.exists():
        print(ad, ": yok"); continue
    t = f.read_text(encoding="utf-8", errors="replace")
    yollar = sorted(set(re.findall(
        r'(?:docs|sonuc|kod|kayit|makale|site)[\\/]+[\w\-. \\/]+', t)))
    print(f"{ad}:")
    for y in yollar[:60]:
        print("   ", y)

print("\n== 2) Bugün 15:30'dan sonra değişen dosyalar ==")
for alt in ["docs", "kod", "sonuc", "kayit", "makale"]:
    d = KOK / alt
    if not d.exists():
        print(f"[{alt}] klasör yok"); continue
    bulunan = []
    for f in d.rglob("*"):
        if f.is_file() and "isci" not in f.parts and f.stat().st_mtime >= BAS:
            bulunan.append((f.stat().st_mtime, f.relative_to(KOK)))
    print(f"[{alt}] {len(bulunan)} dosya")
    for t, f in sorted(bulunan)[:80]:
        print("   ", dt.datetime.fromtimestamp(t).strftime("%H:%M"), f)

AssertionError: Önce site_calisma klasörünün adını site_calisma_eski yap

In [6]:
# Developed with assistance from Claude (Anthropic).
# Site sohbetinin yazdığı dosyaları bulur. Hiçbir şey silmez/değiştirmez.
from pathlib import Path
import re, datetime as dt

KOK = Path(r"C:\Users\USER\orbitrary")
adaylar = [KOK / "site_calisma", KOK / "site_calisma_eski", KOK / "site_calisma_v0"]
SC = next((p for p in adaylar if p.exists()), None)
assert SC is not None, "site klasörü bulunamadı"
print("site klasörü:", SC.name)
BAS = dt.datetime(2026, 11, 14, 15, 30).timestamp()   # site sohbeti ~15:37'de başladı

print("\n== 1) Site dosyalarında geçen yollar ==")
for ad in ["site_adim2.py", "site_adim3.py", "kayit_site_adim3.json", "girdi_raporu.json"]:
    f = SC / ad
    if not f.exists():
        print(ad, ": yok"); continue
    t = f.read_text(encoding="utf-8", errors="replace")
    yollar = sorted(set(re.findall(
        r'(?:docs|sonuc|kod|kayit|makale|site)[\\/]+[\w\-. \\/]+', t)))
    print(f"{ad}:")
    for y in yollar[:60]:
        print("   ", y)

print("\n== 2) Bugün 15:30'dan sonra değişen dosyalar ==")
for alt in ["docs", "kod", "sonuc", "kayit", "makale"]:
    d = KOK / alt
    if not d.exists():
        print(f"[{alt}] klasör yok"); continue
    bulunan = []
    for f in d.rglob("*"):
        if f.is_file() and "isci" not in f.parts and f.stat().st_mtime >= BAS:
            bulunan.append((f.stat().st_mtime, f.relative_to(KOK)))
    print(f"[{alt}] {len(bulunan)} dosya")
    for t, f in sorted(bulunan)[:80]:
        print("   ", dt.datetime.fromtimestamp(t).strftime("%H:%M"), f)

print("\n== 3) Kök klasörde 15:30 sonrası değişenler ==")
for f in KOK.iterdir():
    if f.stat().st_mtime >= BAS:
        print("   ", dt.datetime.fromtimestamp(f.stat().st_mtime).strftime("%H:%M"), f.name)

site klasörü: site_calisma

== 1) Site dosyalarında geçen yollar ==
site_adim2.py:
    docs/data/metin.json şablonu. Boş 
    docs/js/veri.js
    docs\ klasöründe ayrı bir pencerede başlatır
    docs\ ve site_calisma\. kod\
    docs\\
    docs\\ altına yazıldı.
    docs\data\ornek\
    kod\\ayar.json yok
    kod\\hedefler.csv yok
    kod\\izgara.py
    kod\ayar.json
    kod\ayar.json ve kod\hedefler.csv yalnız OKUNUR 
    kod\hedefler.csv
    kod\izgara.py formülü
    kod\izgara.py ile birebir.
    kod\izgara.py toplamları 
    sonuc\ klasörlerine YAZMAZ
site_adim3.py:
    docs/data/metin.json şablonu. Boş 
    docs/data/ornek/
    docs/data/ornek/ satırı eklendi 
    docs/data/ornek/ satırı zaten var.
    docs/data/ornek/ yayına gitmez
    docs/data/ornek/\n
    docs/js/veri.js
    docs\ klasöründe ayrı bir pencerede başlatır
    docs\\
    docs\\ altına yazıldı.
    docs\\data\\analog.json
    docs\\data\\metin.json yazıldı 
    docs\data\analog.json
    docs\data\metin.json
    docs

In [7]:
# Developed with assistance from Claude (Anthropic).
# Eski site denemesinin docs\ dosyalarını arşive TAŞIR (silmez), site_calisma_v1'i açar.
from pathlib import Path
import shutil, datetime as dt

KOK = Path(r"C:\Users\USER\orbitrary")
DOCS = KOK / "docs"
ARS = KOK / "site_v0_arsiv"
BAS = dt.datetime(2026, 11, 14, 15, 30).timestamp()

# 1) docs\ içinde 15:30 sonrası yazılanlar = eski site sohbetinin dosyaları
tasinacak = [f for f in DOCS.rglob("*") if f.is_file() and f.stat().st_mtime >= BAS]
assert len(tasinacak) == 35, f"35 bekleniyordu, {len(tasinacak)} bulundu, DUR"
for f in tasinacak:
    hedef = ARS / f.relative_to(KOK)
    hedef.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(f), str(hedef))
print("arşive taşınan:", len(tasinacak))

# 2) docs\ içinde boş kalan alt klasörleri kaldır (yalnız boş olanlar)
for d in sorted([p for p in DOCS.rglob("*") if p.is_dir()], key=lambda p: -len(p.parts)):
    if not any(d.iterdir()):
        d.rmdir()

# 3) .gitignore: yedeğini al, eski sohbetin eklediği satırı çıkar
gi = KOK / ".gitignore"
shutil.copy2(gi, ARS / ".gitignore_yedek")
satirlar = gi.read_text(encoding="utf-8").splitlines()
yeni = [s for s in satirlar if s.strip() != "docs/data/ornek/"]
print(".gitignore çıkarılan satır:", len(satirlar) - len(yeni))
gi.write_text("\n".join(yeni) + "\n", encoding="utf-8")

# 4) yeni site klasörü
V1 = KOK / "site_calisma_v1"
V1.mkdir(exist_ok=True)

print("\ndocs\\ içinde kalanlar (15:30 öncesi, eski dosyalar):")
kalan = [f for f in DOCS.rglob("*") if f.is_file()]
for f in kalan:
    print("   ", f.relative_to(KOK))
print("toplam kalan:", len(kalan))
print("hazır:", V1)

arşive taşınan: 35
.gitignore çıkarılan satır: 1

docs\ içinde kalanlar (15:30 öncesi, eski dosyalar):
toplam kalan: 0
hazır: C:\Users\USER\orbitrary\site_calisma_v1


In [8]:
%run C:/Users/USER/orbitrary/kod/hucre_kos_d7_v5.py

============================= test session starts =============================
platform win32 -- Python 3.12.16, pytest-9.2.0, pluggy-1.7.0 -- C:\Users\USER\anaconda3\envs\orbitrary\python.exe
rootdir: C:\Users\USER\orbitrary\kod
plugins: anyio-4.16.0, platformdirs-4.14.5
collecting ... collected 10 items

testler/test_kos_hatti.py::test_depo_tam_kosu PASSED                     [ 10%]
testler/test_kos_hatti.py::test_depo_kesilip_devam_K16 PASSED            [ 20%]
testler/test_kos_hatti.py::test_depo_sema_ve_boyut PASSED                [ 30%]
testler/test_kos_hatti.py::test_izgara_sayilari PASSED                   [ 40%]
testler/test_kos_hatti.py::test_depo_iki_isci PASSED                     [ 50%]
testler/test_kos_hatti.py::test_diyagram_deposu PASSED                   [ 60%]
testler/test_kos_puanla.py::test_uzaklik_ve_puanla PASSED                [ 70%]
testler/test_kos_v5_d7.py::test_skor_aday_kumesinden_bagimsiz PASSED     [ 80%]
testler/test_kos_v5_d7.py::test_v5_recall PASSED   

In [9]:
# Developed with assistance from Claude (Anthropic).
from pathlib import Path; import datetime as dt
KOK = Path.home() / "orbitrary"; BAS = dt.datetime(2026, 11, 14, 17, 30).timestamp()
for alt in ["docs", "site_calisma"]:
    for f in (KOK / alt).rglob("*"):
        if f.is_file() and f.stat().st_mtime >= BAS:
            print(dt.datetime.fromtimestamp(f.stat().st_mtime).strftime("%H:%M"), f.relative_to(KOK))
print("bitti")

bitti


In [10]:
%run C:/Users/USER/orbitrary/kod/hucre_kos_katmanb_ozellik.py

============================= test session starts =============================
platform win32 -- Python 3.12.16, pytest-9.2.0, pluggy-1.7.0 -- C:\Users\USER\anaconda3\envs\orbitrary\python.exe
rootdir: C:\Users\USER\orbitrary\kod
plugins: anyio-4.16.0, platformdirs-4.14.5
collecting ... collected 12 items

testler/test_kos_hatti.py::test_depo_tam_kosu PASSED                     [  8%]
testler/test_kos_hatti.py::test_depo_kesilip_devam_K16 PASSED            [ 16%]
testler/test_kos_hatti.py::test_depo_sema_ve_boyut PASSED                [ 25%]
testler/test_kos_hatti.py::test_izgara_sayilari PASSED                   [ 33%]
testler/test_kos_hatti.py::test_depo_iki_isci PASSED                     [ 41%]
testler/test_kos_hatti.py::test_diyagram_deposu PASSED                   [ 50%]
testler/test_kos_puanla.py::test_uzaklik_ve_puanla PASSED                [ 58%]
testler/test_kos_v5_d7.py::test_skor_aday_kumesinden_bagimsiz PASSED     [ 66%]
testler/test_kos_v5_d7.py::test_v5_recall PASSED   

In [11]:
%run C:/Users/USER/orbitrary/kod/hucre_kos_katmanb_puan.py

============================= test session starts =============================
platform win32 -- Python 3.12.16, pytest-9.2.0, pluggy-1.7.0 -- C:\Users\USER\anaconda3\envs\orbitrary\python.exe
rootdir: C:\Users\USER\orbitrary\kod
plugins: anyio-4.16.0, platformdirs-4.14.5
collecting ... collected 15 items

testler/test_kos_hatti.py::test_depo_tam_kosu PASSED                     [  6%]
testler/test_kos_hatti.py::test_depo_kesilip_devam_K16 PASSED            [ 13%]
testler/test_kos_hatti.py::test_depo_sema_ve_boyut PASSED                [ 20%]
testler/test_kos_hatti.py::test_izgara_sayilari PASSED                   [ 26%]
testler/test_kos_hatti.py::test_depo_iki_isci PASSED                     [ 33%]
testler/test_kos_hatti.py::test_diyagram_deposu PASSED                   [ 40%]
testler/test_kos_puanla.py::test_uzaklik_ve_puanla PASSED                [ 46%]
testler/test_kos_v5_d7.py::test_skor_aday_kumesinden_bagimsiz PASSED     [ 53%]
testler/test_kos_v5_d7.py::test_v5_recall PASSED   

In [12]:
%run C:/Users/USER/orbitrary/kod/hucre_kos_kayip.py

============================= test session starts =============================
platform win32 -- Python 3.12.16, pytest-9.2.0, pluggy-1.7.0 -- C:\Users\USER\anaconda3\envs\orbitrary\python.exe
rootdir: C:\Users\USER\orbitrary\kod
plugins: anyio-4.16.0, platformdirs-4.14.5
collecting ... collected 18 items

testler/test_kos_hatti.py::test_depo_tam_kosu PASSED                     [  5%]
testler/test_kos_hatti.py::test_depo_kesilip_devam_K16 PASSED            [ 11%]
testler/test_kos_hatti.py::test_depo_sema_ve_boyut PASSED                [ 16%]
testler/test_kos_hatti.py::test_izgara_sayilari PASSED                   [ 22%]
testler/test_kos_hatti.py::test_depo_iki_isci PASSED                     [ 27%]
testler/test_kos_hatti.py::test_diyagram_deposu PASSED                   [ 33%]
testler/test_kos_puanla.py::test_uzaklik_ve_puanla PASSED                [ 38%]
testler/test_kos_v5_d7.py::test_skor_aday_kumesinden_bagimsiz PASSED     [ 44%]
testler/test_kos_v5_d7.py::test_v5_recall PASSED   

In [13]:
# Developed with assistance from Claude (Anthropic).
# Katman B'de zamanın nereye gittiğini ölçer (30 Mars noktası). Hiçbir sonucu değiştirmez, kaydetmez.
import cProfile, pstats, sys, io
sys.path.insert(0, r"C:\Users\USER\orbitrary\kod")
import kos
pr = cProfile.Profile(); pr.enable()
kos.katmanb_kuru(30, "mars")
pr.disable()
for sira in ("tottime", "cumtime"):
    s = io.StringIO(); pstats.Stats(pr, stream=s).sort_stats(sira).print_stats(25)
    print(f"===== {sira} =====\n" + s.getvalue()[-5000:])

Katman B kuru (mars): kurulum 5 s, 0.92 s/pencere → tek süreç 91.7 sa
===== tottime =====
         5987201 function calls (5846587 primitive calls) in 33.827 seconds

   Ordered by: internal time
   List reduced from 3328 to 25 due to restriction <25>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
       60    5.041    0.084    6.500    0.108 C:\Users\USER\orbitrary\kod\ozellik_topo.py:125(d1)
     5088    2.865    0.001    2.881    0.001 C:\Users\USER\anaconda3\envs\orbitrary\Lib\site-packages\numpy\_core\memmap.py:358(__getitem__)
       41    2.589    0.063    2.589    0.063 {built-in method numpy._core._multiarray_umath._unique_hash}
    58698    2.129    0.000    2.129    0.000 {method 'reduce' of 'numpy.ufunc' objects}
      238    1.877    0.008    2.317    0.010 C:\Users\USER\orbitrary\kod\uzaklik.py:47(w1_matris)
       30    1.465    0.049   15.747    0.525 C:\Users\USER\orbitrary\kod\kos.py:1240(_uzaklik)
    65444    1.284    0.000    1.284    0.00

In [14]:
%run C:/Users/USER/orbitrary/kod/hucre_kos_hiz.py

============================= test session starts =============================
platform win32 -- Python 3.12.16, pytest-9.2.0, pluggy-1.7.0 -- C:\Users\USER\anaconda3\envs\orbitrary\python.exe
rootdir: C:\Users\USER\orbitrary\kod
plugins: anyio-4.16.0, platformdirs-4.14.5
collecting ... collected 26 items

testler/test_kos_hiz.py::test_d1_hizli_birebir PASSED                    [  3%]
testler/test_kos_hiz.py::test_diyagram_tek_ayni PASSED                   [  7%]
testler/test_kos_hiz.py::test_sw_onbellek_ayni PASSED                    [ 11%]
testler/test_ozellik_topoloji.py::test_ayni_diyagram_sw_sifir_8_2 PASSED [ 15%]
testler/test_ozellik_topoloji.py::test_rolyef_sifir_diyagram_bos_8_2 PASSED [ 19%]
testler/test_ozellik_topoloji.py::test_normalizasyon_kappa_taban PASSED  [ 23%]
testler/test_ozellik_topoloji.py::test_iki_cukur_bilinen_diyagram PASSED [ 26%]
testler/test_ozellik_topoloji.py::test_capraz_komsuluk PASSED            [ 30%]
testler/test_ozellik_topoloji.py::test_sw_tek_no

In [15]:
%run C:/Users/USER/orbitrary/kod/hucre_kos_f32.py

============================= test session starts =============================
platform win32 -- Python 3.12.16, pytest-9.2.0, pluggy-1.7.0 -- C:\Users\USER\anaconda3\envs\orbitrary\python.exe
rootdir: C:\Users\USER\orbitrary\kod
plugins: anyio-4.16.0, platformdirs-4.14.5
collecting ... collected 14 items

testler/test_kos_katmanb_puan.py::test_seyrek_tam_ile_ayni PASSED        [  7%]
testler/test_kos_katmanb_puan.py::test_referans_37_sabit_2000 PASSED     [ 14%]
testler/test_kos_katmanb_puan.py::test_kuru_hedef PASSED                 [ 21%]
testler/test_kos_katmanb_puan.py::test_f32_harita PASSED                 [ 28%]
testler/test_kos_hiz.py::test_d1_hizli_birebir PASSED                    [ 35%]
testler/test_kos_hiz.py::test_diyagram_tek_ayni PASSED                   [ 42%]
testler/test_kos_hiz.py::test_sw_onbellek_ayni PASSED                    [ 50%]
testler/test_kos_puanla.py::test_uzaklik_ve_puanla PASSED                [ 57%]
testler/test_kos_hatti.py::test_depo_tam_kosu PASSE

In [16]:
%run C:/Users/USER/orbitrary/kod/hucre_kos_dogrulama.py

============================= test session starts =============================
platform win32 -- Python 3.12.16, pytest-9.2.0, pluggy-1.7.0 -- C:\Users\USER\anaconda3\envs\orbitrary\python.exe
rootdir: C:\Users\USER\orbitrary\kod
plugins: anyio-4.16.0, platformdirs-4.14.5
collecting ... collected 19 items

testler/test_kos_dogrulama.py::test_dogrulama_uctan_uca PASSED           [  5%]
testler/test_kos_puanla.py::test_uzaklik_ve_puanla PASSED                [ 10%]
testler/test_kos_puanla.py::test_puanla_tam_mod PASSED                   [ 15%]
testler/test_dogrula.py::test_pi_ve_K14 PASSED                           [ 21%]
testler/test_dogrula.py::test_fisher_elle PASSED                         [ 26%]
testler/test_dogrula.py::test_fisher_H0_altinda_duzgun PASSED            [ 31%]
testler/test_dogrula.py::test_fisher_yon PASSED                          [ 36%]
testler/test_dogrula.py::test_auc_kaba_kuvvet_ve_mw PASSED               [ 42%]
testler/test_dogrula.py::test_v3_tekrar_ve_GA PASSE

In [17]:
%run C:/Users/USER/orbitrary/kod/hucre_kos_site.py

============================= test session starts =============================
platform win32 -- Python 3.12.16, pytest-9.2.0, pluggy-1.7.0 -- C:\Users\USER\anaconda3\envs\orbitrary\python.exe
rootdir: C:\Users\USER\orbitrary\kod
plugins: anyio-4.16.0, platformdirs-4.14.5
collecting ... collected 4 items

testler/test_kos_site.py::test_site_veri PASSED                          [ 25%]
testler/test_kos_dogrulama.py::test_dogrulama_uctan_uca PASSED           [ 50%]
testler/test_kos_puanla.py::test_uzaklik_ve_puanla PASSED                [ 75%]
testler/test_kos_puanla.py::test_puanla_tam_mod PASSED                   [100%]

============================ slowest 10 durations =============================
6.41s setup    testler/test_kos_puanla.py::test_uzaklik_ve_puanla
4.82s call     testler/test_kos_dogrulama.py::test_dogrulama_uctan_uca
2.83s call     testler/test_kos_site.py::test_site_veri
2.63s call     testler/test_kos_puanla.py::test_puanla_tam_mod
2.11s call     testler/test_kos_pua

In [18]:
# Developed with assistance from Claude (Anthropic).
# Site + site verisini incelemek için tek zip'e koyar (büyük doku/foto/harita dosyaları hariç). Hiçbir dosyayı değiştirmez.
from pathlib import Path
import zipfile, json
KOK = Path.home() / "orbitrary"
SITE = KOK / "site_v2"; VERI = KOK / "sonuc" / "site"
cikti = KOK / "paylas"; cikti.mkdir(exist_ok=True)
zp = cikti / "site_paket.zip"
liste = []
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(SITE.rglob("*")):
        if f.is_dir() or ".ipynb" in f.name or "checkpoint" in str(f):
            continue
        rel = f.relative_to(KOK)
        if f.suffix.lower() in (".jpg", ".jpeg", ".png", ".tif", ".exr", ".webp") or "grid" in f.parts:
            liste.append(f"{rel}  ({f.stat().st_size // 1024} KB, eklenmedi)")
            continue
        z.write(f, rel); liste.append(f"{rel}  ({f.stat().st_size // 1024} KB)")
    if VERI.exists():
        for f in sorted(VERI.rglob("*")):
            if f.is_dir():
                continue
            rel = f.relative_to(KOK)
            if "grid" in f.parts and f.name not in ("0.bin", "136.bin"):
                continue
            z.write(f, rel)
    for ek in ("kayit/SITE_VERI_NOTU.md", "sonuc/dogrulama/dogrulama.json", "kod/site_girdi/analog_metin.csv",
               "kod/site_girdi/mission_prep.csv", "kod/site_girdi/metin.json", "kod/site_girdi/sekil_sude.json"):
        if (KOK / ek).exists():
            z.write(KOK / ek, ek)
    z.writestr("dosya_listesi.txt", "\n".join(liste))
print(f"{zp}  {zp.stat().st_size / 1e6:.1f} MB")
print("site veri klasörü var mı:", VERI.exists())

C:\Users\USER\orbitrary\paylas\site_paket.zip  0.8 MB
site veri klasörü var mı: True


In [1]:
# Developed with assistance from Claude (Anthropic).
import json, collections
from pathlib import Path
L = Path(r"C:\Users\USER\orbitrary\sonuc\liste")
uz = collections.Counter(); kisa = []; tam8 = 0; top = 0; anahtar = None
for p in sorted(L.glob("liste_*.json")):
    o = json.loads(p.read_text(encoding="utf-8"))
    for g, d in o["gorevler"].items():
        if not d or d.get("durum") != "computed" or not d.get("ilk10"): continue
        n = len(d["ilk10"]); uz[n] += 1; top += n
        if n < 10: kisa.append((p.stem[6:], g, n))
        anahtar = anahtar or list(d["ilk10"][0].keys())
        tam8 += sum(all(x is not None for x in r["s"]) for r in d["ilk10"])
print("hesaplanan durum:", sum(uz.values()), "| toplam aday:", top)
print("liste uzunlukları:", dict(uz))
print("10'dan kısa listeler:", kisa)
print("8 kategorisi tam olan aday:", tam8)
print("aday alanları:", anahtar)

hesaplanan durum: 112 | toplam aday: 1091
liste uzunlukları: {10: 100, 8: 4, 9: 4, 7: 2, 5: 1, 4: 1}
10'dan kısa listeler: [('A3-02', 'astro', 8), ('A3-09', 'genel', 9), ('M-A15', 'astro', 7), ('M-SHK', 'geology', 9), ('M-SP', 'geology', 5), ('R-ELY', 'genel', 8), ('R-ELY', 'mobility', 9), ('R-ELY', 'geology', 4), ('R-ELY', 'astro', 7), ('R-PF', 'astro', 8), ('R-PHX', 'astro', 9), ('R-ZHU', 'astro', 8)]
8 kategorisi tam olan aday: 78
aday alanları: ['kimlik', 'enlem', 'boylam', 'P', 'Q', 's', 'd', 'kapsama', 'eksik_taraf', 'kapi_durumu', 'eps1', 'eps1_durum', 'eps2', 'eps2_durum', 'en_yakin_analog']
